# SmokeWatch Replay: Full-Period Data Pipeline

This notebook expands the validated five-day feasibility workflow to the complete Dixie Fire pilot period from July 13 through August 31, 2021.

The first step creates a complete UTC county-hour backbone. This structural grid ensures that zero fire detections remain distinct from missing PM2.5 or weather measurements.

In [20]:
from pathlib import Path

import pandas as pd
import numpy as np

# Locate the repository by searching the current directory
# and its parent directories for the Git repository.
current_directory = Path.cwd().resolve()
project_root = None

for candidate_directory in [
    current_directory,
    *current_directory.parents,
]:
    if (
        (candidate_directory / ".git").exists()
        and (candidate_directory / "notebooks").is_dir()
        and (candidate_directory / "data").is_dir()
    ):
        project_root = candidate_directory
        break

if project_root is None:
    raise RuntimeError(
        "Could not locate the smokewatch-replay project root."
    )

pilot_counties = [
    "Butte",
    "Lassen",
    "Plumas",
    "Shasta",
    "Tehama",
]

pilot_start_utc = pd.Timestamp(
    "2021-07-13 00:00:00",
    tz="UTC",
)

pilot_end_utc = pd.Timestamp(
    "2021-08-31 23:00:00",
    tz="UTC",
)

expected_hour_count = 1200
expected_county_count = 5
expected_county_hour_rows = 6000

full_period_interim_directory = (
    project_root / "data" / "interim" / "full_period"
)
full_period_interim_directory.mkdir(
    parents=True,
    exist_ok=True,
)

county_hour_grid_path = (
    full_period_interim_directory
    / "dixie_county_hour_grid_2021-07-13_2021-08-31.csv"
)

print("Notebook working directory:", current_directory)
print("Project root:", project_root)
print("Pilot start:", pilot_start_utc)
print("Pilot end:", pilot_end_utc)
print("Pilot counties:", pilot_counties)
print("Expected hours:", expected_hour_count)
print("Expected county-hour rows:", expected_county_hour_rows)

Notebook working directory: /Users/VV/Documents/GitHub/smokewatch-replay/notebooks
Project root: /Users/VV/Documents/GitHub/smokewatch-replay
Pilot start: 2021-07-13 00:00:00+00:00
Pilot end: 2021-08-31 23:00:00+00:00
Pilot counties: ['Butte', 'Lassen', 'Plumas', 'Shasta', 'Tehama']
Expected hours: 1200
Expected county-hour rows: 6000


In [21]:
full_period_hours = pd.date_range(
    start=pilot_start_utc,
    end=pilot_end_utc,
    freq="h",
)

full_county_hour_grid = (
    pd.MultiIndex.from_product(
        [full_period_hours, pilot_counties],
        names=["event_hour_utc", "county"],
    )
    .to_frame(index=False)
    .sort_values(["event_hour_utc", "county"])
    .reset_index(drop=True)
)

print("UTC hours generated:", len(full_period_hours))
print("Counties:", len(pilot_counties))
print("County-hour rows:", len(full_county_hour_grid))

display(full_county_hour_grid.head(10))
display(full_county_hour_grid.tail(10))

UTC hours generated: 1200
Counties: 5
County-hour rows: 6000


,event_hour_utc,county
0,2021-07-13 00:00:00+00:00,Butte
1,2021-07-13 00:00:00+00:00,Lassen
2,2021-07-13 00:00:00+00:00,Plumas
3,2021-07-13 00:00:00+00:00,Shasta
4,2021-07-13 00:00:00+00:00,Tehama
5,2021-07-13 01:00:00+00:00,Butte
6,2021-07-13 01:00:00+00:00,Lassen
7,2021-07-13 01:00:00+00:00,Plumas
8,2021-07-13 01:00:00+00:00,Shasta
9,2021-07-13 01:00:00+00:00,Tehama


,event_hour_utc,county
5990,2021-08-31 22:00:00+00:00,Butte
5991,2021-08-31 22:00:00+00:00,Lassen
5992,2021-08-31 22:00:00+00:00,Plumas
5993,2021-08-31 22:00:00+00:00,Shasta
5994,2021-08-31 22:00:00+00:00,Tehama
5995,2021-08-31 23:00:00+00:00,Butte
5996,2021-08-31 23:00:00+00:00,Lassen
5997,2021-08-31 23:00:00+00:00,Plumas
5998,2021-08-31 23:00:00+00:00,Shasta
5999,2021-08-31 23:00:00+00:00,Tehama


In [22]:
rows_per_county = (
    full_county_hour_grid
    .groupby("county", as_index=False)
    .agg(
        county_hour_rows=("event_hour_utc", "size"),
        unique_hours=("event_hour_utc", "nunique"),
        earliest_hour=("event_hour_utc", "min"),
        latest_hour=("event_hour_utc", "max"),
    )
)

rows_per_hour = (
    full_county_hour_grid
    .groupby("event_hour_utc")
    .size()
)

continuous_hourly_sequence = (
    full_period_hours
    .to_series()
    .diff()
    .dropna()
    .eq(pd.Timedelta(hours=1))
    .all()
)

grid_validation = pd.DataFrame(
    [
        (
            "Total county-hour rows",
            len(full_county_hour_grid),
            expected_county_hour_rows,
        ),
        (
            "Unique UTC hours",
            full_county_hour_grid["event_hour_utc"].nunique(),
            expected_hour_count,
        ),
        (
            "Unique counties",
            full_county_hour_grid["county"].nunique(),
            expected_county_count,
        ),
        (
            "Duplicate county-hour keys",
            full_county_hour_grid.duplicated(
                subset=["event_hour_utc", "county"]
            ).sum(),
            0,
        ),
        (
            "Invalid timestamps",
            full_county_hour_grid["event_hour_utc"].isna().sum(),
            0,
        ),
        (
            "Minimum county rows per hour",
            rows_per_hour.min(),
            expected_county_count,
        ),
        (
            "Maximum county rows per hour",
            rows_per_hour.max(),
            expected_county_count,
        ),
        (
            "Minimum hours per county",
            rows_per_county["county_hour_rows"].min(),
            expected_hour_count,
        ),
        (
            "Maximum hours per county",
            rows_per_county["county_hour_rows"].max(),
            expected_hour_count,
        ),
        (
            "Continuous hourly sequence",
            continuous_hourly_sequence,
            True,
        ),
        (
            "Timestamp timezone is UTC",
            str(
                full_county_hour_grid[
                    "event_hour_utc"
                ].dt.tz
            ),
            "UTC",
        ),
    ],
    columns=["check", "observed", "expected"],
)

grid_validation["status"] = grid_validation.apply(
    lambda row: (
        "PASS"
        if row["observed"] == row["expected"]
        else "FAIL"
    ),
    axis=1,
)

display(grid_validation)
display(rows_per_county)

failed_grid_checks = grid_validation[
    grid_validation["status"] != "PASS"
]

assert failed_grid_checks.empty, (
    "The full-period grid failed validation:\n"
    + failed_grid_checks.to_string(index=False)
)

assert (
    full_county_hour_grid["event_hour_utc"].min()
    == pilot_start_utc
)

assert (
    full_county_hour_grid["event_hour_utc"].max()
    == pilot_end_utc
)

print("Full-period county-hour grid validation passed.")

,check,observed,expected,status
0,Total county-hour rows,6000,6000,PASS
1,Unique UTC hours,1200,1200,PASS
2,Unique counties,5,5,PASS
3,Duplicate county-hour keys,0,0,PASS
4,Invalid timestamps,0,0,PASS
5,Minimum county rows per hour,5,5,PASS
6,Maximum county rows per hour,5,5,PASS
7,Minimum hours per county,1200,1200,PASS
8,Maximum hours per county,1200,1200,PASS
9,Continuous hourly sequence,True,True,PASS


,county,county_hour_rows,unique_hours,earliest_hour,latest_hour
0,Butte,1200,1200,2021-07-13 00:00:00+00:00,2021-08-31 23:00:00+00:00
1,Lassen,1200,1200,2021-07-13 00:00:00+00:00,2021-08-31 23:00:00+00:00
2,Plumas,1200,1200,2021-07-13 00:00:00+00:00,2021-08-31 23:00:00+00:00
3,Shasta,1200,1200,2021-07-13 00:00:00+00:00,2021-08-31 23:00:00+00:00
4,Tehama,1200,1200,2021-07-13 00:00:00+00:00,2021-08-31 23:00:00+00:00


Full-period county-hour grid validation passed.


In [23]:
full_county_hour_grid.to_csv(
    county_hour_grid_path,
    index=False,
)

reloaded_county_hour_grid = pd.read_csv(
    county_hour_grid_path
)

reloaded_county_hour_grid["event_hour_utc"] = (
    pd.to_datetime(
        reloaded_county_hour_grid["event_hour_utc"],
        utc=True,
        errors="coerce",
    )
)

assert len(reloaded_county_hour_grid) == 6000

assert (
    reloaded_county_hour_grid["event_hour_utc"]
    .isna()
    .sum()
    == 0
)

assert (
    reloaded_county_hour_grid.duplicated(
        subset=["event_hour_utc", "county"]
    ).sum()
    == 0
)

assert (
    reloaded_county_hour_grid["county"].nunique()
    == 5
)

assert (
    reloaded_county_hour_grid["event_hour_utc"].nunique()
    == 1200
)

pd.testing.assert_frame_equal(
    full_county_hour_grid,
    reloaded_county_hour_grid,
)

print("Full-period grid saved and reloaded successfully.")
print("Output path:", county_hour_grid_path)
print("Reloaded rows:", len(reloaded_county_hour_grid))
print("Reloaded columns:", len(reloaded_county_hour_grid.columns))

Full-period grid saved and reloaded successfully.
Output path: /Users/VV/Documents/GitHub/smokewatch-replay/data/interim/full_period/dixie_county_hour_grid_2021-07-13_2021-08-31.csv
Reloaded rows: 6000
Reloaded columns: 2


## Full-Period Structural Backbone Validation

The complete pilot period contains 1,200 UTC hours across five counties, producing 6,000 unique county-hour rows.

This grid will serve as the structural backbone for the full-period NASA FIRMS, EPA AirData, and NOAA GHCNh datasets. Each source will be validated independently before it is joined to this backbone. Missing measurements will remain explicitly missing, while valid zero-event records will remain zero.

## Full-Period NASA FIRMS Ingestion

NASA FIRMS limits an Area API request to a maximum of five days. Therefore, the 50-day pilot period is divided into ten non-overlapping five-day requests.

The pipeline uses the same `VIIRS_NOAA20_SP` standard-processing product and California bounding box as the feasibility sample. The private FIRMS MAP_KEY is loaded from `.env` and is never printed, displayed, or saved in the notebook.

In [24]:
import os
import time

import requests
from dotenv import load_dotenv

load_dotenv(
    dotenv_path=project_root / ".env",
    override=False,
)

firms_map_key = os.getenv("FIRMS_MAP_KEY")

if not firms_map_key:
    raise RuntimeError(
        "FIRMS_MAP_KEY was not found in the repository-root .env file."
    )

firms_product = "VIIRS_NOAA20_SP"

# West, south, east, north
california_bbox = "-125,32,-113,43"

firms_api_base = (
    "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
)

firms_raw_directory = (
    project_root
    / "data"
    / "raw"
    / "nasa_firms"
    / "full_period"
)
firms_raw_directory.mkdir(
    parents=True,
    exist_ok=True,
)

combined_firms_raw_path = (
    firms_raw_directory
    / "viirs_noaa20_sp_california_2021-07-13_2021-08-31.csv"
)

firms_manifest_path = (
    firms_raw_directory
    / "viirs_noaa20_sp_download_manifest_2021-07-13_2021-08-31.csv"
)

chunk_start_dates = pd.date_range(
    start=pilot_start_utc.normalize(),
    end=pilot_end_utc.normalize(),
    freq="5D",
)

manifest_records = []

for chunk_number, chunk_start in enumerate(
    chunk_start_dates,
    start=1,
):
    chunk_end = min(
        chunk_start + pd.Timedelta(days=4),
        pilot_end_utc.normalize(),
    )

    day_range = (
        chunk_end.normalize()
        - chunk_start.normalize()
    ).days + 1

    chunk_filename = (
        "viirs_noaa20_sp_california_"
        f"{chunk_start:%Y-%m-%d}_"
        f"{chunk_end:%Y-%m-%d}.csv"
    )

    manifest_records.append(
        {
            "chunk_number": chunk_number,
            "chunk_start_date": chunk_start.date(),
            "chunk_end_date": chunk_end.date(),
            "day_range": day_range,
            "output_filename": chunk_filename,
        }
    )

firms_request_manifest = pd.DataFrame(
    manifest_records
)

assert len(firms_request_manifest) == 10
assert firms_request_manifest["day_range"].between(1, 5).all()
assert firms_request_manifest["day_range"].sum() == 50
assert (
    firms_request_manifest["chunk_start_date"].iloc[0]
    == pilot_start_utc.date()
)
assert (
    firms_request_manifest["chunk_end_date"].iloc[-1]
    == pilot_end_utc.date()
)

print("FIRMS key loaded securely:", bool(firms_map_key))
print("Product:", firms_product)
print("Bounding box:", california_bbox)
print("Request chunks:", len(firms_request_manifest))
print(
    "Total calendar days:",
    firms_request_manifest["day_range"].sum(),
)

display(firms_request_manifest)

FIRMS key loaded securely: True
Product: VIIRS_NOAA20_SP
Bounding box: -125,32,-113,43
Request chunks: 10
Total calendar days: 50


,chunk_number,chunk_start_date,chunk_end_date,day_range,output_filename
0,1,2021-07-13,2021-07-17,5,viirs_noaa20_sp_california_2021-07-13_2021-07-...
1,2,2021-07-18,2021-07-22,5,viirs_noaa20_sp_california_2021-07-18_2021-07-...
2,3,2021-07-23,2021-07-27,5,viirs_noaa20_sp_california_2021-07-23_2021-07-...
3,4,2021-07-28,2021-08-01,5,viirs_noaa20_sp_california_2021-07-28_2021-08-...
4,5,2021-08-02,2021-08-06,5,viirs_noaa20_sp_california_2021-08-02_2021-08-...
5,6,2021-08-07,2021-08-11,5,viirs_noaa20_sp_california_2021-08-07_2021-08-...
6,7,2021-08-12,2021-08-16,5,viirs_noaa20_sp_california_2021-08-12_2021-08-...
7,8,2021-08-17,2021-08-21,5,viirs_noaa20_sp_california_2021-08-17_2021-08-...
8,9,2021-08-22,2021-08-26,5,viirs_noaa20_sp_california_2021-08-22_2021-08-...
9,10,2021-08-27,2021-08-31,5,viirs_noaa20_sp_california_2021-08-27_2021-08-...


In [25]:
required_firms_columns = {
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
    "confidence",
    "type",
    "frp",
    "daynight",
}

def read_and_validate_firms_chunk(
    chunk_path,
    chunk_start_date,
    chunk_end_date,
):
    chunk_frame = pd.read_csv(
        chunk_path,
        dtype={"acq_time": "string"},
    )

    missing_columns = (
        required_firms_columns
        - set(chunk_frame.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{chunk_path.name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    parsed_dates = pd.to_datetime(
        chunk_frame["acq_date"],
        errors="coerce",
    )

    if parsed_dates.isna().any():
        raise ValueError(
            f"{chunk_path.name} contains invalid acquisition dates."
        )

    earliest_allowed = pd.Timestamp(chunk_start_date)
    latest_allowed = pd.Timestamp(chunk_end_date)

    outside_chunk = ~parsed_dates.between(
        earliest_allowed,
        latest_allowed,
    )

    if outside_chunk.any():
        raise ValueError(
            f"{chunk_path.name} contains records outside "
            "its requested date range."
        )

    return chunk_frame


firms_chunk_frames = []
download_log = []

for request_row in firms_request_manifest.itertuples(
    index=False
):
    chunk_path = (
        firms_raw_directory
        / request_row.output_filename
    )

    if chunk_path.exists():
        download_status = "reused existing file"
    else:
        request_url = (
            f"{firms_api_base}/"
            f"{firms_map_key}/"
            f"{firms_product}/"
            f"{california_bbox}/"
            f"{request_row.day_range}/"
            f"{request_row.chunk_start_date}"
        )

        response = requests.get(
            request_url,
            timeout=120,
        )
        response.raise_for_status()

        response_text = response.text

        if not response_text.lstrip().startswith(
            "latitude,longitude"
        ):
            raise RuntimeError(
                "A FIRMS response did not contain the expected "
                f"CSV header for chunk {request_row.chunk_number}."
            )

        chunk_path.write_text(
            response_text,
            encoding="utf-8",
        )

        download_status = "downloaded"
        time.sleep(0.5)

    chunk_frame = read_and_validate_firms_chunk(
        chunk_path=chunk_path,
        chunk_start_date=request_row.chunk_start_date,
        chunk_end_date=request_row.chunk_end_date,
    )

    firms_chunk_frames.append(chunk_frame)

    download_log.append(
        {
            "chunk_number": request_row.chunk_number,
            "chunk_start_date": request_row.chunk_start_date,
            "chunk_end_date": request_row.chunk_end_date,
            "status": download_status,
            "record_count": len(chunk_frame),
            "column_count": len(chunk_frame.columns),
            "file_size_bytes": chunk_path.stat().st_size,
        }
    )

firms_download_summary = pd.DataFrame(
    download_log
)

display(firms_download_summary)

print(
    "Chunks successfully validated:",
    len(firms_download_summary),
)
print(
    "Total downloaded chunk records:",
    firms_download_summary["record_count"].sum(),
)

,chunk_number,chunk_start_date,chunk_end_date,status,record_count,column_count,file_size_bytes
0,1,2021-07-13,2021-07-17,reused existing file,9604,15,784562
1,2,2021-07-18,2021-07-22,reused existing file,11672,15,956307
2,3,2021-07-23,2021-07-27,reused existing file,8029,15,659056
3,4,2021-07-28,2021-08-01,reused existing file,2171,15,178312
4,5,2021-08-02,2021-08-06,reused existing file,17692,15,1447014
5,6,2021-08-07,2021-08-11,reused existing file,14320,15,1174729
6,7,2021-08-12,2021-08-16,reused existing file,14099,15,1156342
7,8,2021-08-17,2021-08-21,reused existing file,29668,15,2427883
8,9,2021-08-22,2021-08-26,reused existing file,15629,15,1281843
9,10,2021-08-27,2021-08-31,reused existing file,21095,15,1732405


Chunks successfully validated: 10
Total downloaded chunk records: 143979


In [26]:
full_period_firms_raw = (
    pd.concat(
        firms_chunk_frames,
        ignore_index=True,
    )
)

full_period_firms_raw["acq_date_parsed"] = pd.to_datetime(
    full_period_firms_raw["acq_date"],
    errors="coerce",
)

raw_source_columns = [
    column
    for column in full_period_firms_raw.columns
    if column != "acq_date_parsed"
]

exact_duplicate_count = (
    full_period_firms_raw[raw_source_columns]
    .duplicated()
    .sum()
)

outside_pilot_period = ~full_period_firms_raw[
    "acq_date_parsed"
].between(
    pilot_start_utc.tz_localize(None).normalize(),
    pilot_end_utc.tz_localize(None).normalize(),
)

invalid_coordinate_count = (
    full_period_firms_raw["latitude"].isna()
    | full_period_firms_raw["longitude"].isna()
    | ~full_period_firms_raw["latitude"].between(
        32,
        43,
    )
    | ~full_period_firms_raw["longitude"].between(
        -125,
        -113,
    )
).sum()

daily_raw_counts = (
    full_period_firms_raw
    .groupby("acq_date", as_index=False)
    .size()
    .rename(columns={"size": "record_count"})
)

full_period_firms_raw[
    raw_source_columns
].to_csv(
    combined_firms_raw_path,
    index=False,
)

firms_download_summary.to_csv(
    firms_manifest_path,
    index=False,
)

raw_firms_validation = pd.DataFrame(
    [
        (
            "Request chunks",
            len(firms_download_summary),
            10,
        ),
        (
            "Raw records",
            len(full_period_firms_raw),
            "greater than 0",
        ),
        (
            "Source columns",
            len(raw_source_columns),
            15,
        ),
        (
            "Invalid acquisition dates",
            full_period_firms_raw[
                "acq_date_parsed"
            ].isna().sum(),
            0,
        ),
        (
            "Records outside pilot period",
            outside_pilot_period.sum(),
            0,
        ),
        (
            "Invalid California coordinates",
            invalid_coordinate_count,
            0,
        ),
        (
            "Exact duplicate records",
            exact_duplicate_count,
            0,
        ),
    ],
    columns=["check", "observed", "expected"],
)

raw_firms_validation["status"] = [
    (
        "PASS"
        if row.check == "Raw records"
        and row.observed > 0
        else (
            "PASS"
            if row.observed == row.expected
            else "FAIL"
        )
    )
    for row in raw_firms_validation.itertuples()
]

display(raw_firms_validation)
display(daily_raw_counts.head())
display(daily_raw_counts.tail())

assert (
    raw_firms_validation["status"] == "PASS"
).all()

print("Combined raw FIRMS file saved successfully.")
print("Output path:", combined_firms_raw_path)
print("Combined rows:", len(full_period_firms_raw))
print("Source columns:", len(raw_source_columns))
print(
    "Earliest acquisition date:",
    full_period_firms_raw["acq_date_parsed"].min(),
)
print(
    "Latest acquisition date:",
    full_period_firms_raw["acq_date_parsed"].max(),
)

,check,observed,expected,status
0,Request chunks,10,10,PASS
1,Raw records,143979,greater than 0,PASS
2,Source columns,15,15,PASS
3,Invalid acquisition dates,0,0,PASS
4,Records outside pilot period,0,0,PASS
5,Invalid California coordinates,0,0,PASS
6,Exact duplicate records,0,0,PASS


,acq_date,record_count
0,2021-07-13,1910
1,2021-07-14,1112
2,2021-07-15,1834
3,2021-07-16,1898
4,2021-07-17,2850


,acq_date,record_count
45,2021-08-27,2637
46,2021-08-28,3039
47,2021-08-29,4488
48,2021-08-30,5443
49,2021-08-31,5488


Combined raw FIRMS file saved successfully.
Output path: /Users/VV/Documents/GitHub/smokewatch-replay/data/raw/nasa_firms/full_period/viirs_noaa20_sp_california_2021-07-13_2021-08-31.csv
Combined rows: 143979
Source columns: 15
Earliest acquisition date: 2021-07-13 00:00:00
Latest acquisition date: 2021-08-31 00:00:00


## NASA FIRMS Filtering and Incident-Area Assignment

The full-period FIRMS records are processed using the same rules validated during feasibility:

- retain vegetation-fire records where `type = 0`;
- retain nominal or high confidence (`n` or `h`);
- assign records to the five pilot counties;
- retain records inside the final Dixie Fire perimeter with a 1 km buffer;
- normalize acquisition timestamps to UTC;
- aggregate qualifying detections to county-hour;
- preserve zero-detection county-hours through the complete 6,000-row grid.

The five-day overlap must reproduce the previously validated NASA sample before the full-period output is accepted.

In [27]:
five_day_reference_raw_path = (
    project_root
    / "data"
    / "raw"
    / "nasa_firms"
    / "viirs_noaa20_california_2021-07-13_5days.csv"
)

if not five_day_reference_raw_path.exists():
    raise FileNotFoundError(
        "The validated five-day FIRMS raw file was not found."
    )

five_day_reference_raw = pd.read_csv(
    five_day_reference_raw_path,
    dtype={"acq_time": "string"},
)

first_full_period_chunk = pd.read_csv(
    firms_raw_directory
    / firms_request_manifest.loc[
        0,
        "output_filename",
    ],
    dtype={"acq_time": "string"},
)

assert set(five_day_reference_raw.columns) == set(
    first_full_period_chunk.columns
)

comparison_columns = list(
    five_day_reference_raw.columns
)

reference_unique = (
    five_day_reference_raw[comparison_columns]
    .drop_duplicates()
)

current_chunk_unique = (
    first_full_period_chunk[comparison_columns]
    .drop_duplicates()
)

reference_comparison = reference_unique.merge(
    current_chunk_unique,
    on=comparison_columns,
    how="left",
    indicator=True,
)

reference_rows_missing_from_current = (
    reference_comparison["_merge"]
    .eq("left_only")
    .sum()
)

current_comparison = current_chunk_unique.merge(
    reference_unique,
    on=comparison_columns,
    how="left",
    indicator=True,
)

additional_current_rows = (
    current_comparison["_merge"]
    .eq("left_only")
    .sum()
)

raw_overlap_summary = pd.DataFrame(
    {
        "measure": [
            "Validated five-day raw rows",
            "Current first-chunk rows",
            "Reference rows missing from current chunk",
            "Additional rows in current chunk",
        ],
        "value": [
            len(five_day_reference_raw),
            len(first_full_period_chunk),
            reference_rows_missing_from_current,
            additional_current_rows,
        ],
    }
)

display(raw_overlap_summary)

print(
    "Raw overlap comparison complete. "
    "The processed incident-area overlap will be "
    "the final reproducibility test."
)

,measure,value
0,Validated five-day raw rows,2956
1,Current first-chunk rows,9604
2,Reference rows missing from current chunk,0
3,Additional rows in current chunk,6648


Raw overlap comparison complete. The processed incident-area overlap will be the final reproducibility test.


In [28]:
import geopandas as gpd

county_boundary_url = (
    "https://www2.census.gov/geo/tiger/"
    "GENZ2021/shp/cb_2021_us_county_500k.zip"
)

county_boundary_path = (
    project_root
    / "data"
    / "raw"
    / "reference"
    / "cb_2021_us_county_500k.zip"
)

county_boundary_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not county_boundary_path.exists():
    county_response = requests.get(
        county_boundary_url,
        timeout=120,
    )
    county_response.raise_for_status()

    county_boundary_path.write_bytes(
        county_response.content
    )

calfire_perimeter_path = (
    project_root
    / "data"
    / "raw"
    / "reference"
    / "calfire_dixie_2021_final_perimeter.geojson"
)

if not calfire_perimeter_path.exists():
    raise FileNotFoundError(
        "The validated CAL FIRE Dixie perimeter file "
        "was not found."
    )

firms_filter_input = full_period_firms_raw[
    raw_source_columns
].copy()

firms_filter_input["type"] = pd.to_numeric(
    firms_filter_input["type"],
    errors="coerce",
)

firms_filter_input["confidence"] = (
    firms_filter_input["confidence"]
    .astype("string")
    .str.lower()
)

firms_filter_input["frp"] = pd.to_numeric(
    firms_filter_input["frp"],
    errors="coerce",
)

firms_filter_input["latitude"] = pd.to_numeric(
    firms_filter_input["latitude"],
    errors="coerce",
)

firms_filter_input["longitude"] = pd.to_numeric(
    firms_filter_input["longitude"],
    errors="coerce",
)

firms_filter_input["acq_time_text"] = (
    firms_filter_input["acq_time"]
    .astype("string")
    .str.zfill(4)
)

firms_filter_input["event_timestamp_utc"] = (
    pd.to_datetime(
        (
            firms_filter_input["acq_date"].astype(str)
            + " "
            + firms_filter_input["acq_time_text"]
        ),
        format="%Y-%m-%d %H%M",
        utc=True,
        errors="coerce",
    )
)

quality_filtered_firms = firms_filter_input[
    firms_filter_input["type"].eq(0)
    & firms_filter_input["confidence"].isin(["n", "h"])
    & firms_filter_input["event_timestamp_utc"].notna()
    & firms_filter_input["latitude"].notna()
    & firms_filter_input["longitude"].notna()
    & firms_filter_input["frp"].notna()
].copy()

county_boundaries = gpd.read_file(
    county_boundary_path,
    engine="pyogrio",
)

pilot_county_boundaries = (
    county_boundaries[
        county_boundaries["STATEFP"].astype(str).eq("06")
        & county_boundaries["NAME"].isin(pilot_counties)
    ][["NAME", "geometry"]]
    .rename(columns={"NAME": "county"})
    .copy()
)

assert len(pilot_county_boundaries) == 5
assert pilot_county_boundaries["county"].nunique() == 5

firms_points = gpd.GeoDataFrame(
    quality_filtered_firms,
    geometry=gpd.points_from_xy(
        quality_filtered_firms["longitude"],
        quality_filtered_firms["latitude"],
    ),
    crs="EPSG:4326",
)

firms_points_in_county_crs = firms_points.to_crs(
    pilot_county_boundaries.crs
)

pilot_county_firms = gpd.sjoin(
    firms_points_in_county_crs,
    pilot_county_boundaries,
    how="inner",
    predicate="within",
).drop(columns=["index_right"])

calfire_perimeter = gpd.read_file(
    calfire_perimeter_path,
    engine="pyogrio",
)

if calfire_perimeter.crs is None:
    raise ValueError(
        "The CAL FIRE perimeter has no defined CRS."
    )

calfire_perimeter_projected = (
    calfire_perimeter.to_crs("EPSG:3310")
)

dixie_buffer_geometry = (
    calfire_perimeter_projected
    .geometry
    .union_all()
    .buffer(1000)
)

dixie_buffer = gpd.GeoDataFrame(
    {"incident": ["Dixie Fire"]},
    geometry=[dixie_buffer_geometry],
    crs="EPSG:3310",
)

pilot_county_firms_projected = (
    pilot_county_firms.to_crs("EPSG:3310")
)

full_period_dixie_firms = gpd.sjoin(
    pilot_county_firms_projected,
    dixie_buffer,
    how="inner",
    predicate="within",
).drop(columns=["index_right"])

full_period_dixie_firms["event_hour_utc"] = (
    full_period_dixie_firms["event_timestamp_utc"]
    .dt.floor("h")
)

filter_summary = pd.DataFrame(
    {
        "stage": [
            "Raw California-bbox records",
            "Quality-filtered records",
            "Records inside pilot counties",
            "Records inside buffered Dixie perimeter",
        ],
        "record_count": [
            len(firms_filter_input),
            len(quality_filtered_firms),
            len(pilot_county_firms),
            len(full_period_dixie_firms),
        ],
    }
)

display(filter_summary)

print(
    "Pilot counties represented:",
    full_period_dixie_firms["county"].nunique(),
)
print(
    "Invalid event timestamps:",
    full_period_dixie_firms[
        "event_timestamp_utc"
    ].isna().sum(),
)

,stage,record_count
0,Raw California-bbox records,143979
1,Quality-filtered records,136113
2,Records inside pilot counties,57884
3,Records inside buffered Dixie perimeter,53394


Pilot counties represented: 5
Invalid event timestamps: 0


In [29]:
full_period_fire_aggregated = (
    full_period_dixie_firms
    .groupby(
        ["event_hour_utc", "county"],
        as_index=False,
    )
    .agg(
        fire_detection_count=(
            "latitude",
            "size",
        ),
        total_frp_mw=(
            "frp",
            "sum",
        ),
        mean_frp_mw=(
            "frp",
            "mean",
        ),
        max_frp_mw=(
            "frp",
            "max",
        ),
        high_confidence_count=(
            "confidence",
            lambda values: values.eq("h").sum(),
        ),
        night_detection_count=(
            "daynight",
            lambda values: (
                values.astype("string")
                .str.upper()
                .eq("N")
                .sum()
            ),
        ),
    )
)

full_period_fire_hourly = (
    full_county_hour_grid
    .merge(
        full_period_fire_aggregated,
        on=["event_hour_utc", "county"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(["event_hour_utc", "county"])
    .reset_index(drop=True)
)

count_columns = [
    "fire_detection_count",
    "high_confidence_count",
    "night_detection_count",
]

for column in count_columns:
    full_period_fire_hourly[column] = (
        full_period_fire_hourly[column]
        .fillna(0)
        .astype("int64")
    )

full_period_fire_hourly["total_frp_mw"] = (
    full_period_fire_hourly["total_frp_mw"]
    .fillna(0.0)
)

fire_county_summary = (
    full_period_fire_hourly
    .groupby("county", as_index=False)
    .agg(
        total_hours=("event_hour_utc", "size"),
        detection_hours=(
            "fire_detection_count",
            lambda values: values.gt(0).sum(),
        ),
        total_fire_detections=(
            "fire_detection_count",
            "sum",
        ),
        total_frp_mw=("total_frp_mw", "sum"),
        high_confidence_detections=(
            "high_confidence_count",
            "sum",
        ),
        night_detections=(
            "night_detection_count",
            "sum",
        ),
    )
)

display(fire_county_summary)

print(
    "Full-period fire-grid rows:",
    len(full_period_fire_hourly),
)
print(
    "County-hours with detections:",
    full_period_fire_hourly[
        "fire_detection_count"
    ].gt(0).sum(),
)
print(
    "Total qualifying detections:",
    full_period_fire_hourly[
        "fire_detection_count"
    ].sum(),
)

,county,total_hours,detection_hours,total_fire_detections,total_frp_mw,high_confidence_detections,night_detections
0,Butte,1200,44,1090,21648.54,163,648
1,Lassen,1200,80,10107,144156.89,788,6846
2,Plumas,1200,142,36538,716066.66,3756,25945
3,Shasta,1200,54,3766,61547.95,337,2416
4,Tehama,1200,66,1893,33420.87,144,1373


Full-period fire-grid rows: 6000
County-hours with detections: 386
Total qualifying detections: 53394


In [30]:
five_day_fire_sample_path = (
    project_root
    / "data"
    / "interim"
    / "nasa_firms"
    / "dixie_fire_hourly_county_sample_2021-07-13_17.csv"
)

if not five_day_fire_sample_path.exists():
    raise FileNotFoundError(
        "The validated five-day NASA interim file was not found."
    )

validated_five_day_fire = pd.read_csv(
    five_day_fire_sample_path
)

validated_five_day_fire["event_hour_utc"] = (
    pd.to_datetime(
        validated_five_day_fire["event_hour_utc"],
        utc=True,
        errors="coerce",
    )
)

full_period_five_day_overlap = (
    full_period_fire_hourly[
        full_period_fire_hourly["event_hour_utc"].between(
            pd.Timestamp(
                "2021-07-13 00:00:00",
                tz="UTC",
            ),
            pd.Timestamp(
                "2021-07-17 23:00:00",
                tz="UTC",
            ),
        )
    ]
    .reset_index(drop=True)
)

validated_five_day_fire = (
    validated_five_day_fire[
        full_period_five_day_overlap.columns
    ]
    .sort_values(["event_hour_utc", "county"])
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    full_period_five_day_overlap,
    validated_five_day_fire,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

full_period_fire_output_path = (
    project_root
    / "data"
    / "interim"
    / "nasa_firms"
    / "dixie_fire_hourly_county_full_2021-07-13_2021-08-31.csv"
)

full_period_fire_hourly.to_csv(
    full_period_fire_output_path,
    index=False,
)

reloaded_full_period_fire = pd.read_csv(
    full_period_fire_output_path
)

reloaded_full_period_fire["event_hour_utc"] = (
    pd.to_datetime(
        reloaded_full_period_fire["event_hour_utc"],
        utc=True,
        errors="coerce",
    )
)

assert len(reloaded_full_period_fire) == 6000

assert (
    reloaded_full_period_fire.duplicated(
        subset=["event_hour_utc", "county"]
    ).sum()
    == 0
)

assert (
    reloaded_full_period_fire[
        "fire_detection_count"
    ].sum()
    == len(full_period_dixie_firms)
)

assert (
    full_period_five_day_overlap[
        "fire_detection_count"
    ].sum()
    == 976
)

assert (
    full_period_five_day_overlap[
        "fire_detection_count"
    ].gt(0).sum()
    == 19
)

print("Five-day NASA overlap reproduced successfully.")
print("Overlap detections:", 976)
print("Overlap detection hours:", 19)
print("Full-period NASA output saved successfully.")
print("Output path:", full_period_fire_output_path)
print("Saved rows:", len(reloaded_full_period_fire))
print("Saved columns:", len(reloaded_full_period_fire.columns))

Five-day NASA overlap reproduced successfully.
Overlap detections: 976
Overlap detection hours: 19
Full-period NASA output saved successfully.
Output path: /Users/VV/Documents/GitHub/smokewatch-replay/data/interim/nasa_firms/dixie_fire_hourly_county_full_2021-07-13_2021-08-31.csv
Saved rows: 6000
Saved columns: 8


### Full-Period NASA FIRMS Validation Result

The full-period NASA FIRMS pipeline successfully reproduced the validated five-day sample before expanding to the complete pilot period.

The combined download contained 143,979 raw records. After applying the vegetation-fire and confidence rules, 136,113 records remained. County assignment retained 57,884 records, and the buffered Dixie Fire perimeter retained 53,394 qualifying detections.

The final county-hour dataset contains:

- 6,000 structural county-hour rows;
- 386 county-hours with one or more qualifying detections;
- 53,394 total qualifying detections;
- 5,444 high-confidence detections;
- 37,228 nighttime detections;
- zero duplicate county-hour keys;
- zero invalid event timestamps.

The validated July 13–17 overlap reproduced all 976 detections and 19 detection county-hours exactly. This confirms that the expanded pipeline preserves the feasibility-stage transformation rules.

Zero-detection county-hours remain in the dataset with a detection count of zero. Mean and maximum FRP remain missing for these hours because no detection-level FRP measurement exists.

## Full-Period EPA AirData PM2.5 Pipeline

The full-period EPA pipeline reuses the validated 2021 annual archives for parameter products 88101 and 88502.

Records are filtered to the five pilot counties and the July 13–August 31 UTC period. Multiple instruments are first collapsed to site-hour values, followed by county-product-hour aggregation. Product 88101 is preferred when both products are available.

Raw PM2.5 measurements are preserved. Negative measurements are retained in `pm25_ug_m3_raw`, adjusted to zero only in the cleaned `pm25_ug_m3` field, and explicitly flagged.

In [31]:
epa_88101_path = (
    project_root
    / "data"
    / "raw"
    / "epa_airdata"
    / "hourly_88101_2021.zip"
)

epa_88502_path = (
    project_root
    / "data"
    / "raw"
    / "epa_airdata"
    / "hourly_88502_2021.zip"
)

for required_path in [
    epa_88101_path,
    epa_88502_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required EPA archive not found: {required_path}"
        )

epa_use_columns = [
    "State Code",
    "County Code",
    "Site Num",
    "POC",
    "County Name",
    "Date GMT",
    "Time GMT",
    "Sample Measurement",
    "Units of Measure",
    "Qualifier",
]

def load_full_period_epa_archive(
    archive_path,
    product_code,
):
    retained_frames = []
    scanned_rows = 0

    archive_reader = pd.read_csv(
        archive_path,
        compression="zip",
        usecols=epa_use_columns,
        dtype={
            "State Code": "string",
            "County Code": "string",
            "Site Num": "string",
            "POC": "string",
        },
        chunksize=250_000,
        low_memory=False,
    )

    for chunk in archive_reader:
        scanned_rows += len(chunk)

        state_code = (
            chunk["State Code"]
            .astype("string")
            .str.zfill(2)
        )

        county_mask = chunk["County Name"].isin(
            pilot_counties
        )

        state_mask = state_code.eq("06")

        retained_chunk = chunk[
            state_mask & county_mask
        ].copy()

        if retained_chunk.empty:
            continue

        retained_chunk["event_hour_utc"] = (
            pd.to_datetime(
                (
                    retained_chunk["Date GMT"].astype(str)
                    + " "
                    + retained_chunk["Time GMT"].astype(str)
                ),
                utc=True,
                errors="coerce",
            ).dt.floor("h")
        )

        retained_chunk = retained_chunk[
            retained_chunk["event_hour_utc"].between(
                pilot_start_utc,
                pilot_end_utc,
            )
        ].copy()

        if retained_chunk.empty:
            continue

        retained_chunk["sample_measurement"] = (
            pd.to_numeric(
                retained_chunk["Sample Measurement"],
                errors="coerce",
            )
        )

        retained_chunk["qualifier_text"] = (
            retained_chunk["Qualifier"]
            .fillna("")
            .astype("string")
            .str.strip()
        )

        retained_chunk["wildfire_flagged"] = (
            retained_chunk["qualifier_text"]
            .str.contains(
                r"(?:^|[;,\s])IT(?:$|[;,\s])",
                regex=True,
                na=False,
            )
        )

        retained_chunk["county"] = (
            retained_chunk["County Name"]
            .astype("string")
        )

        retained_chunk["state_code"] = (
            retained_chunk["State Code"]
            .astype("string")
            .str.zfill(2)
        )

        retained_chunk["county_code"] = (
            retained_chunk["County Code"]
            .astype("string")
            .str.zfill(3)
        )

        retained_chunk["site_number"] = (
            retained_chunk["Site Num"]
            .astype("string")
            .str.zfill(4)
        )

        retained_chunk["site_id"] = (
            retained_chunk["state_code"]
            + "-"
            + retained_chunk["county_code"]
            + "-"
            + retained_chunk["site_number"]
        )

        retained_chunk["pm25_product"] = str(
            product_code
        )

        retained_frames.append(
            retained_chunk[
                [
                    "event_hour_utc",
                    "county",
                    "site_id",
                    "POC",
                    "pm25_product",
                    "sample_measurement",
                    "qualifier_text",
                    "wildfire_flagged",
                    "Units of Measure",
                ]
            ]
        )

    if retained_frames:
        retained_data = pd.concat(
            retained_frames,
            ignore_index=True,
        )
    else:
        retained_data = pd.DataFrame(
            columns=[
                "event_hour_utc",
                "county",
                "site_id",
                "POC",
                "pm25_product",
                "sample_measurement",
                "qualifier_text",
                "wildfire_flagged",
                "Units of Measure",
            ]
        )

    return retained_data, scanned_rows


epa_88101_full_raw, epa_88101_scanned = (
    load_full_period_epa_archive(
        epa_88101_path,
        "88101",
    )
)

epa_88502_full_raw, epa_88502_scanned = (
    load_full_period_epa_archive(
        epa_88502_path,
        "88502",
    )
)

epa_load_summary = pd.DataFrame(
    {
        "product": ["88101", "88502"],
        "annual_rows_scanned": [
            epa_88101_scanned,
            epa_88502_scanned,
        ],
        "full_period_rows_retained": [
            len(epa_88101_full_raw),
            len(epa_88502_full_raw),
        ],
        "monitor_sites": [
            epa_88101_full_raw["site_id"].nunique(),
            epa_88502_full_raw["site_id"].nunique(),
        ],
    }
)

display(epa_load_summary)

,product,annual_rows_scanned,full_period_rows_retained,monitor_sites
0,88101,8620507,2385,2
1,88502,2397085,6853,6


In [32]:
epa_full_raw = pd.concat(
    [
        epa_88101_full_raw,
        epa_88502_full_raw,
    ],
    ignore_index=True,
)

five_day_end_utc = pd.Timestamp(
    "2021-07-17 23:00:00",
    tz="UTC",
)

five_day_epa_raw = epa_full_raw[
    epa_full_raw["event_hour_utc"].between(
        pilot_start_utc,
        five_day_end_utc,
    )
].copy()

five_day_product_counts = (
    five_day_epa_raw
    .groupby("pm25_product", as_index=False)
    .size()
    .rename(columns={"size": "record_count"})
)

qualifier_summary = (
    epa_full_raw
    .assign(
        displayed_qualifier=lambda frame: (
            frame["qualifier_text"]
            .replace("", "(blank)")
        )
    )
    .groupby(
        ["pm25_product", "displayed_qualifier"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "record_count"})
)

unit_summary = (
    epa_full_raw
    .groupby(
        ["pm25_product", "Units of Measure"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "record_count"})
)

display(five_day_product_counts)
display(qualifier_summary)
display(unit_summary)

five_day_count_lookup = dict(
    zip(
        five_day_product_counts["pm25_product"],
        five_day_product_counts["record_count"],
    )
)

assert five_day_count_lookup.get("88101") == 238
assert five_day_count_lookup.get("88502") == 703

assert (
    epa_full_raw["event_hour_utc"].isna().sum()
    == 0
)

assert (
    epa_full_raw["event_hour_utc"].between(
        pilot_start_utc,
        pilot_end_utc,
    ).all()
)

print("Five-day EPA raw overlap reproduced successfully.")
print("Product 88101 overlap rows:", 238)
print("Product 88502 overlap rows:", 703)
print("Full-period retained raw rows:", len(epa_full_raw))
print("Full-period monitor sites:", epa_full_raw["site_id"].nunique())

,pm25_product,record_count
0,88101,238
1,88502,703


,pm25_product,displayed_qualifier,record_count
0,88101,(blank),1770
1,88101,IT,615
2,88502,(blank),2932
3,88502,IJ,71
4,88502,IT,3850


,pm25_product,Units of Measure,record_count
0,88101,Micrograms/cubic meter (LC),2385
1,88502,Micrograms/cubic meter (LC),6853


Five-day EPA raw overlap reproduced successfully.
Product 88101 overlap rows: 238
Product 88502 overlap rows: 703
Full-period retained raw rows: 9238
Full-period monitor sites: 8


In [33]:
epa_measurements = epa_full_raw[
    epa_full_raw["sample_measurement"].notna()
].copy()

epa_site_hour = (
    epa_measurements
    .groupby(
        [
            "event_hour_utc",
            "county",
            "pm25_product",
            "site_id",
        ],
        as_index=False,
    )
    .agg(
        pm25_site_ug_m3=(
            "sample_measurement",
            "mean",
        ),
        wildfire_flagged=(
            "wildfire_flagged",
            "max",
        ),
        instrument_count=(
            "POC",
            "nunique",
        ),
    )
)

epa_county_product_hour = (
    epa_site_hour
    .groupby(
        [
            "event_hour_utc",
            "county",
            "pm25_product",
        ],
        as_index=False,
    )
    .agg(
        pm25_ug_m3_raw=(
            "pm25_site_ug_m3",
            "mean",
        ),
        monitor_site_count=(
            "site_id",
            "nunique",
        ),
        wildfire_flagged_site_count=(
            "wildfire_flagged",
            "sum",
        ),
    )
)

product_priority = {
    "88101": 1,
    "88502": 2,
}

epa_county_product_hour["product_priority"] = (
    epa_county_product_hour["pm25_product"]
    .map(product_priority)
)

epa_preferred_county_hour = (
    epa_county_product_hour
    .sort_values(
        [
            "event_hour_utc",
            "county",
            "product_priority",
        ]
    )
    .drop_duplicates(
        subset=["event_hour_utc", "county"],
        keep="first",
    )
    .drop(columns=["product_priority"])
    .reset_index(drop=True)
)

full_period_pm25 = (
    full_county_hour_grid
    .merge(
        epa_preferred_county_hour,
        on=["event_hour_utc", "county"],
        how="left",
        validate="one_to_one",
    )
)

full_period_pm25["pm25_available"] = (
    full_period_pm25["pm25_ug_m3_raw"]
    .notna()
)

full_period_pm25["pm25_negative_adjusted"] = (
    full_period_pm25["pm25_ug_m3_raw"]
    .lt(0)
    .fillna(False)
)

full_period_pm25["pm25_ug_m3"] = (
    full_period_pm25["pm25_ug_m3_raw"]
    .clip(lower=0)
)

# Preserve missing site counts when no PM2.5 measurement exists.
# A missing value is different from a confirmed count of zero.
for column in [
    "monitor_site_count",
    "wildfire_flagged_site_count",
]:
    full_period_pm25[column] = pd.to_numeric(
        full_period_pm25[column],
        errors="coerce",
    )

epa_output_columns = [
    "event_hour_utc",
    "county",
    "pm25_ug_m3_raw",
    "pm25_ug_m3",
    "pm25_negative_adjusted",
    "monitor_site_count",
    "wildfire_flagged_site_count",
    "pm25_product",
    "pm25_available",
]

full_period_pm25 = (
    full_period_pm25[epa_output_columns]
    .sort_values(["event_hour_utc", "county"])
    .reset_index(drop=True)
)

pm25_county_summary = (
    full_period_pm25
    .groupby("county", as_index=False)
    .agg(
        total_hours=("event_hour_utc", "size"),
        available_hours=("pm25_available", "sum"),
        missing_hours=(
            "pm25_available",
            lambda values: (~values).sum(),
        ),
        negative_adjustments=(
            "pm25_negative_adjusted",
            "sum",
        ),
        maximum_pm25_ug_m3=(
            "pm25_ug_m3",
            "max",
        ),
    )
)

display(pm25_county_summary)

print("EPA site-hour rows:", len(epa_site_hour))
print(
    "EPA county-product-hour rows:",
    len(epa_county_product_hour),
)
print(
    "Preferred county-hour measurements:",
    len(epa_preferred_county_hour),
)

,county,total_hours,available_hours,missing_hours,negative_adjustments,maximum_pm25_ug_m3
0,Butte,1200,1200,0,0,218.0
1,Lassen,1200,0,1200,0,NaN
2,Plumas,1200,1200,0,1,424.0
3,Shasta,1200,1197,3,0,459.0
4,Tehama,1200,1194,6,0,292.0


EPA site-hour rows: 9238
EPA county-product-hour rows: 5981
Preferred county-hour measurements: 4791


In [36]:
five_day_pm25_sample_path = (
    project_root
    / "data"
    / "interim"
    / "epa_airdata"
    / "dixie_pm25_hourly_county_sample_2021-07-13_17.csv"
)

if not five_day_pm25_sample_path.exists():
    raise FileNotFoundError(
        "The validated five-day EPA interim file was not found."
    )

validated_five_day_pm25 = pd.read_csv(
    five_day_pm25_sample_path,
    dtype={"pm25_product": "string"},
)

validated_five_day_pm25["event_hour_utc"] = (
    pd.to_datetime(
        validated_five_day_pm25["event_hour_utc"],
        utc=True,
        errors="coerce",
    )
)

for boolean_column in [
    "pm25_negative_adjusted",
    "pm25_available",
]:
    validated_five_day_pm25[boolean_column] = (
        validated_five_day_pm25[boolean_column]
        .astype("string")
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
            }
        )
    )

full_period_five_day_pm25 = (
    full_period_pm25[
        full_period_pm25["event_hour_utc"].between(
            pilot_start_utc,
            five_day_end_utc,
        )
    ]
    .sort_values(["event_hour_utc", "county"])
    .reset_index(drop=True)
)

validated_five_day_pm25 = (
    validated_five_day_pm25[
        full_period_five_day_pm25.columns
    ]
    .sort_values(["event_hour_utc", "county"])
    .reset_index(drop=True)
)

# Give missing products the same temporary comparison value.
# This avoids the pd.NA-versus-NaN comparison warning.
for comparison_frame in [
    full_period_five_day_pm25,
    validated_five_day_pm25,
]:
    comparison_frame["pm25_product"] = (
        comparison_frame["pm25_product"]
        .astype("string")
        .fillna("__MISSING_PRODUCT__")
    )

pd.testing.assert_frame_equal(
    full_period_five_day_pm25,
    validated_five_day_pm25,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

full_period_pm25_output_path = (
    project_root
    / "data"
    / "interim"
    / "epa_airdata"
    / "dixie_pm25_hourly_county_full_2021-07-13_2021-08-31.csv"
)

full_period_pm25.to_csv(
    full_period_pm25_output_path,
    index=False,
)

reloaded_full_period_pm25 = pd.read_csv(
    full_period_pm25_output_path,
    dtype={"pm25_product": "string"},
)

reloaded_full_period_pm25["event_hour_utc"] = (
    pd.to_datetime(
        reloaded_full_period_pm25["event_hour_utc"],
        utc=True,
        errors="coerce",
    )
)

assert len(reloaded_full_period_pm25) == 6000

assert (
    reloaded_full_period_pm25.duplicated(
        subset=["event_hour_utc", "county"]
    ).sum()
    == 0
)

assert (
    reloaded_full_period_pm25[
        "pm25_ug_m3"
    ]
    .dropna()
    .lt(0)
    .sum()
    == 0
)

assert (
    full_period_five_day_pm25[
        "pm25_available"
    ].sum()
    == 480
)

assert (
    full_period_five_day_pm25[
        "pm25_negative_adjusted"
    ].sum()
    == 1
)

print("Five-day EPA overlap reproduced successfully.")
print("Overlap PM2.5 available hours:", 480)
print("Overlap negative adjustments:", 1)
print("Full-period EPA output saved successfully.")
print("Output path:", full_period_pm25_output_path)
print("Saved rows:", len(reloaded_full_period_pm25))
print("Saved columns:", len(reloaded_full_period_pm25.columns))

Five-day EPA overlap reproduced successfully.
Overlap PM2.5 available hours: 480
Overlap negative adjustments: 1
Full-period EPA output saved successfully.
Output path: /Users/VV/Documents/GitHub/smokewatch-replay/data/interim/epa_airdata/dixie_pm25_hourly_county_full_2021-07-13_2021-08-31.csv
Saved rows: 6000
Saved columns: 9


/var/folders/_1/rzw3yvgx72bflktk01sxjkc40000gp/T/ipykernel_5365/1758341376.py:74: FutureWarning: Mismatched null-like values <NA> and nan found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  pd.testing.assert_frame_equal(


### Full-Period EPA PM2.5 Validation Result

The full-period EPA pipeline successfully reproduced the validated five-day PM2.5 sample.

The annual archives contained 8,620,507 product-88101 records and 2,397,085 product-88502 records. Filtering to the pilot counties and period retained 9,238 measurements across eight monitoring sites.

After site-hour aggregation, county-product aggregation, and product preference:

- 4,791 county-hours contain PM2.5 measurements;
- Butte and Plumas have complete 1,200-hour coverage;
- Shasta has 1,197 available hours;
- Tehama has 1,194 available hours;
- Lassen has no PM2.5 coverage and remains explicitly missing;
- one negative Plumas measurement is preserved in the raw field and adjusted only in the cleaned field;
- the final dataset contains 6,000 structural rows and nine columns.

The July 13–17 overlap reproduced all 480 available PM2.5 county-hours and the single negative-value adjustment exactly. Missing site counts remain missing rather than being converted to zero.

## Full-Period NOAA GHCNh Weather Pipeline

The full-period weather pipeline reuses the validated annual GHCNh station files and station assignments.

Lassen uses Susanville Municipal as its selected station. Plumas uses the same Susanville station as an explicitly labeled nearby proxy because the initially selected Portola file did not contain the required weather fields.

The first step loads the complete pilot-period observations and inspects field availability and NOAA quality codes before cleaning or hourly aggregation.

In [37]:
weather_station_assignments = pd.DataFrame(
    [
        {
            "county": "Butte",
            "weather_station_id": "USW00093210",
            "weather_station_name": "Oroville Municipal Airport",
            "station_assignment": "in_county",
        },
        {
            "county": "Lassen",
            "weather_station_id": "USI0000KSVE",
            "weather_station_name": "Susanville Municipal Airport",
            "station_assignment": "in_county",
        },
        {
            "county": "Plumas",
            "weather_station_id": "USI0000KSVE",
            "weather_station_name": "Susanville Municipal Airport",
            "station_assignment": "nearby_proxy",
        },
        {
            "county": "Shasta",
            "weather_station_id": "USW00024257",
            "weather_station_name": "Redding Municipal Airport",
            "station_assignment": "in_county",
        },
        {
            "county": "Tehama",
            "weather_station_id": "USW00024216",
            "weather_station_name": "Red Bluff Municipal Airport",
            "station_assignment": "in_county",
        },
    ]
)

noaa_raw_directory = (
    project_root
    / "data"
    / "raw"
    / "noaa_ghcnh"
)

def locate_station_file(station_id):
    expected_path = (
        noaa_raw_directory
        / f"GHCNh_{station_id}_2021.psv"
    )

    if expected_path.exists():
        return expected_path

    matching_paths = list(
        noaa_raw_directory.glob(
            f"*{station_id}*2021*.psv"
        )
    )

    if len(matching_paths) == 1:
        return matching_paths[0]

    raise FileNotFoundError(
        f"Could not uniquely locate the 2021 GHCNh file "
        f"for station {station_id}. Matches: "
        f"{matching_paths}"
    )

weather_station_assignments["raw_file_path"] = (
    weather_station_assignments[
        "weather_station_id"
    ].map(locate_station_file)
)

display(weather_station_assignments)

,county,weather_station_id,weather_station_name,station_assignment,raw_file_path
0,Butte,USW00093210,Oroville Municipal Airport,in_county,/Users/VV/Documents/GitHub/smokewatch-replay/d...
1,Lassen,USI0000KSVE,Susanville Municipal Airport,in_county,/Users/VV/Documents/GitHub/smokewatch-replay/d...
2,Plumas,USI0000KSVE,Susanville Municipal Airport,nearby_proxy,/Users/VV/Documents/GitHub/smokewatch-replay/d...
3,Shasta,USW00024257,Redding Municipal Airport,in_county,/Users/VV/Documents/GitHub/smokewatch-replay/d...
4,Tehama,USW00024216,Red Bluff Municipal Airport,in_county,/Users/VV/Documents/GitHub/smokewatch-replay/d...


In [38]:
required_weather_columns = [
    "DATE",
    "temperature",
    "relative_humidity",
    "wind_speed",
    "wind_direction",
]

optional_weather_columns = [
    "temperature_Measurement_Code",
    "temperature_Quality_Code",
    "relative_humidity_Measurement_Code",
    "relative_humidity_Quality_Code",
    "wind_speed_Measurement_Code",
    "wind_speed_Quality_Code",
    "wind_direction_Measurement_Code",
    "wind_direction_Quality_Code",
]

station_file_cache = {}
station_file_summary_records = []

for station_id in (
    weather_station_assignments[
        "weather_station_id"
    ].unique()
):
    station_path = locate_station_file(station_id)

    station_header = pd.read_csv(
        station_path,
        sep="|",
        nrows=0,
    )

    missing_required_columns = (
        set(required_weather_columns)
        - set(station_header.columns)
    )

    if missing_required_columns:
        raise ValueError(
            f"{station_id} is missing required columns: "
            f"{sorted(missing_required_columns)}"
        )

    selected_weather_columns = (
        required_weather_columns
        + [
            column
            for column in optional_weather_columns
            if column in station_header.columns
        ]
    )

    station_data = pd.read_csv(
        station_path,
        sep="|",
        usecols=selected_weather_columns,
        low_memory=False,
    )

    raw_station_rows = len(station_data)

    station_data["event_timestamp_utc"] = (
        pd.to_datetime(
            station_data["DATE"],
            utc=True,
            errors="coerce",
        )
    )

    station_data = station_data[
        station_data["event_timestamp_utc"].between(
            pilot_start_utc,
            pilot_end_utc
            + pd.Timedelta(minutes=59, seconds=59),
        )
    ].copy()

    station_file_cache[station_id] = station_data

    station_file_summary_records.append(
        {
            "weather_station_id": station_id,
            "raw_annual_rows": raw_station_rows,
            "pilot_period_rows": len(station_data),
            "invalid_pilot_timestamps": (
                station_data[
                    "event_timestamp_utc"
                ].isna().sum()
            ),
            "available_columns": len(
                selected_weather_columns
            ),
        }
    )

station_file_summary = pd.DataFrame(
    station_file_summary_records
)

assigned_weather_frames = []

for assignment in (
    weather_station_assignments.itertuples(
        index=False
    )
):
    assigned_frame = station_file_cache[
        assignment.weather_station_id
    ].copy()

    assigned_frame["county"] = assignment.county
    assigned_frame["weather_station_id"] = (
        assignment.weather_station_id
    )
    assigned_frame["weather_station_name"] = (
        assignment.weather_station_name
    )
    assigned_frame["station_assignment"] = (
        assignment.station_assignment
    )

    assigned_weather_frames.append(
        assigned_frame
    )

full_period_weather_raw = pd.concat(
    assigned_weather_frames,
    ignore_index=True,
)

five_day_weather_raw = full_period_weather_raw[
    full_period_weather_raw[
        "event_timestamp_utc"
    ].between(
        pilot_start_utc,
        five_day_end_utc
        + pd.Timedelta(minutes=59, seconds=59),
    )
].copy()

five_day_weather_counts = (
    five_day_weather_raw
    .groupby("county", as_index=False)
    .size()
    .rename(columns={"size": "record_count"})
)

display(station_file_summary)
display(five_day_weather_counts)

expected_five_day_weather_counts = {
    "Butte": 120,
    "Lassen": 360,
    "Plumas": 360,
    "Shasta": 140,
    "Tehama": 140,
}

observed_five_day_weather_counts = dict(
    zip(
        five_day_weather_counts["county"],
        five_day_weather_counts["record_count"],
    )
)

assert (
    observed_five_day_weather_counts
    == expected_five_day_weather_counts
)

print(
    "Five-day raw weather overlap reproduced:",
    len(five_day_weather_raw),
)
print(
    "Full-period assigned weather records:",
    len(full_period_weather_raw),
)

,weather_station_id,raw_annual_rows,pilot_period_rows,invalid_pilot_timestamps,available_columns
0,USW00093210,10400,1288,0,13
1,USI0000KSVE,25280,3296,0,13
2,USW00024257,12127,1555,0,13
3,USW00024216,11693,1482,0,13


,county,record_count
0,Butte,120
1,Lassen,360
2,Plumas,360
3,Shasta,140
4,Tehama,140


Five-day raw weather overlap reproduced: 1120
Full-period assigned weather records: 10917


In [39]:
weather_variables = [
    "temperature",
    "relative_humidity",
    "wind_speed",
    "wind_direction",
]

quality_summary_records = []

for variable in weather_variables:
    quality_column = f"{variable}_Quality_Code"

    if quality_column in full_period_weather_raw.columns:
        quality_counts = (
            full_period_weather_raw[
                quality_column
            ]
            .fillna("")
            .astype("string")
            .str.strip()
            .replace("", "(blank)")
            .value_counts(dropna=False)
        )

        for quality_code, record_count in (
            quality_counts.items()
        ):
            quality_summary_records.append(
                {
                    "variable": variable,
                    "quality_code": quality_code,
                    "record_count": record_count,
                }
            )
    else:
        quality_summary_records.append(
            {
                "variable": variable,
                "quality_code": "(column absent)",
                "record_count": len(
                    full_period_weather_raw
                ),
            }
        )

weather_quality_summary = pd.DataFrame(
    quality_summary_records
)

weather_value_diagnostics = []

for variable in weather_variables:
    numeric_values = pd.to_numeric(
        full_period_weather_raw[variable],
        errors="coerce",
    )

    weather_value_diagnostics.append(
        {
            "variable": variable,
            "total_records": len(numeric_values),
            "numeric_values": numeric_values.notna().sum(),
            "missing_or_non_numeric": numeric_values.isna().sum(),
            "minimum": numeric_values.min(),
            "maximum": numeric_values.max(),
        }
    )

weather_value_diagnostics = pd.DataFrame(
    weather_value_diagnostics
)

wind_direction_numeric = pd.to_numeric(
    full_period_weather_raw["wind_direction"],
    errors="coerce",
)

wind_direction_special_count = (
    wind_direction_numeric.eq(999).sum()
)

display(weather_quality_summary)
display(weather_value_diagnostics)

print(
    "Raw NOAA 999 wind-direction reports:",
    wind_direction_special_count,
)
print(
    "Proxy-assigned raw records:",
    full_period_weather_raw[
        "station_assignment"
    ].eq("nearby_proxy").sum(),
)

,variable,quality_code,record_count
0,temperature,1.0,6580
1,temperature,5,2636
2,temperature,5.0,1286
3,temperature,1,396
4,temperature,(blank),12
5,temperature,A,5
6,temperature,2.0,2
7,relative_humidity,(blank),10915
8,relative_humidity,f,2
9,wind_speed,1.0,6562


,variable,total_records,numeric_values,missing_or_non_numeric,minimum,maximum
0,temperature,10917,10905,12,6.0,46.1
1,relative_humidity,10917,10905,12,3.0,81.0
2,wind_speed,10917,10866,51,0.0,14.9
3,wind_direction,10917,10866,51,10.0,999.0


Raw NOAA 999 wind-direction reports: 2530
Proxy-assigned raw records: 3296


## Clean and aggregate full-period NOAA weather observations

GHCNh quality codes depend on the original data source. This pipeline applies
source-aware quality handling, preserves accepted or edited observations, removes
suspect/error observations, handles `999` wind direction as a special report,
and aggregates valid observations into UTC county-hour weather features.

In [41]:
weather_clean = full_period_weather_raw.copy()

# Normalize the mixed-case NOAA column names.
weather_clean.columns = [
    str(column).strip().lower()
    for column in weather_clean.columns
]

required_weather_columns = {
    "county",
    "event_timestamp_utc",
    "temperature",
    "temperature_quality_code",
    "relative_humidity",
    "relative_humidity_quality_code",
    "wind_speed",
    "wind_speed_quality_code",
    "wind_direction",
    "wind_direction_quality_code",
}

missing_weather_columns = (
    required_weather_columns
    - set(weather_clean.columns)
)

if missing_weather_columns:
    raise KeyError(
        "Required NOAA columns are missing: "
        f"{sorted(missing_weather_columns)}"
    )


def normalize_noaa_code(value):
    """Normalize NOAA quality codes while preserving letter case."""
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.lower() in {"", "nan", "none", "<na>"}:
        return ""

    # Convert values such as 1.0 and 5.0 to 1 and 5.
    try:
        numeric_value = float(text)

        if numeric_value.is_integer():
            return str(int(numeric_value))
    except ValueError:
        pass

    return text


def noaa_quality_is_acceptable(quality_code):
    """
    Keep quality codes validated in the earlier five-day pipeline.

    Accepted numeric codes:
    0, 1, 4, 5, and 9

    Accepted uppercase codes:
    A, U, P, I, M, and R

    Lowercase codes and suspect/error numeric codes are rejected.
    """
    code = normalize_noaa_code(quality_code)

    if code == "":
        return True

    accepted_numeric_codes = {
        "0",
        "1",
        "4",
        "5",
        "9",
    }

    accepted_uppercase_codes = {
        "A",
        "U",
        "P",
        "I",
        "M",
        "R",
    }

    return (
        code in accepted_numeric_codes
        or code in accepted_uppercase_codes
    )


weather_variables = [
    "temperature",
    "relative_humidity",
    "wind_speed",
    "wind_direction",
]

for variable in weather_variables:
    weather_clean[f"{variable}_numeric"] = pd.to_numeric(
        weather_clean[variable],
        errors="coerce",
    )

    weather_clean[f"{variable}_quality_ok"] = (
        weather_clean[f"{variable}_quality_code"]
        .map(noaa_quality_is_acceptable)
    )

# Physical plausibility checks.
weather_clean["temperature_physical_ok"] = (
    weather_clean["temperature_numeric"].between(-80, 60)
)

weather_clean["relative_humidity_physical_ok"] = (
    weather_clean["relative_humidity_numeric"].between(0, 100)
)

weather_clean["wind_speed_physical_ok"] = (
    weather_clean["wind_speed_numeric"].between(0, 100)
)

weather_clean["wind_direction_physical_ok"] = (
    weather_clean["wind_direction_numeric"].between(0, 360)
    | weather_clean["wind_direction_numeric"].eq(999)
)

# Create cleaned measurement columns.
weather_clean["temperature_clean"] = (
    weather_clean["temperature_numeric"].where(
        weather_clean["temperature_quality_ok"]
        & weather_clean["temperature_physical_ok"]
    )
)

weather_clean["relative_humidity_clean"] = (
    weather_clean["relative_humidity_numeric"].where(
        weather_clean["relative_humidity_quality_ok"]
        & weather_clean["relative_humidity_physical_ok"]
    )
)

weather_clean["wind_speed_clean"] = (
    weather_clean["wind_speed_numeric"].where(
        weather_clean["wind_speed_quality_ok"]
        & weather_clean["wind_speed_physical_ok"]
    )
)

# NOAA uses 999 when wind direction is variable or not reportable.
weather_clean["wind_direction_is_special"] = (
    weather_clean["wind_direction_numeric"].eq(999)
    & weather_clean["wind_direction_quality_ok"]
)

weather_clean["wind_direction_clean"] = (
    weather_clean["wind_direction_numeric"].where(
        weather_clean["wind_direction_quality_ok"]
        & weather_clean["wind_direction_numeric"].between(0, 360)
    )
)

# Wind direction 360 degrees is equivalent to 0 degrees.
weather_clean.loc[
    weather_clean["wind_direction_clean"].eq(360),
    "wind_direction_clean",
] = 0.0

quality_summary_rows = []
quality_rejection_rows = []

for variable in weather_variables:
    numeric_column = f"{variable}_numeric"
    quality_ok_column = f"{variable}_quality_ok"
    physical_ok_column = f"{variable}_physical_ok"
    quality_column = f"{variable}_quality_code"

    numeric_present = weather_clean[numeric_column].notna()

    quality_rejected = (
        numeric_present
        & ~weather_clean[quality_ok_column]
    )

    physical_rejected = (
        numeric_present
        & weather_clean[quality_ok_column]
        & ~weather_clean[physical_ok_column]
    )

    quality_summary_rows.append(
        {
            "variable": variable,
            "input_records": len(weather_clean),
            "numeric_values": int(numeric_present.sum()),
            "missing_values": int((~numeric_present).sum()),
            "quality_rejected": int(quality_rejected.sum()),
            "physical_rejected": int(physical_rejected.sum()),
        }
    )

    if quality_rejected.any():
        rejection_detail = (
            weather_clean.loc[
                quality_rejected,
                [quality_column],
            ]
            .fillna("<blank>")
            .value_counts()
            .rename("record_count")
            .reset_index()
        )

        rejection_detail.insert(0, "variable", variable)

        rejection_detail.columns = [
            "variable",
            "quality_code",
            "record_count",
        ]

        quality_rejection_rows.append(rejection_detail)

weather_quality_summary = pd.DataFrame(
    quality_summary_rows
)

display(weather_quality_summary)

if quality_rejection_rows:
    weather_quality_rejections = pd.concat(
        quality_rejection_rows,
        ignore_index=True,
    )

    display(weather_quality_rejections)
else:
    print("No NOAA observations were rejected by quality code.")

print(
    "Accepted uppercase A temperature records:",
    int(
        weather_clean[
            "temperature_quality_code"
        ]
        .astype("string")
        .str.strip()
        .eq("A")
        .sum()
    ),
)

print(
    "Raw wind-direction 999 reports:",
    int(weather_clean["wind_direction_is_special"].sum()),
)

/var/folders/_1/rzw3yvgx72bflktk01sxjkc40000gp/T/ipykernel_5365/129829357.py:212: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("<blank>")


,variable,input_records,numeric_values,missing_values,quality_rejected,physical_rejected
0,temperature,10917,10905,12,2,0
1,relative_humidity,10917,10905,12,2,0
2,wind_speed,10917,10866,51,0,0
3,wind_direction,10917,10866,51,0,0


,variable,quality_code,record_count
0,temperature,2.0,2
1,relative_humidity,f,2


Accepted uppercase A temperature records: 5
Raw wind-direction 999 reports: 2530


In [42]:
def circular_mean_degrees(values):
    """Calculate a circular mean for wind directions in degrees."""
    valid_values = pd.to_numeric(
        values,
        errors="coerce",
    ).dropna()

    if valid_values.empty:
        return np.nan

    angles_radians = np.deg2rad(
        valid_values.to_numpy(dtype=float)
    )

    mean_sine = np.sin(angles_radians).mean()
    mean_cosine = np.cos(angles_radians).mean()

    if np.isclose(mean_sine, 0) and np.isclose(mean_cosine, 0):
        return np.nan

    mean_degrees = (
        np.degrees(np.arctan2(mean_sine, mean_cosine))
        + 360
    ) % 360

    if np.isclose(mean_degrees, 360):
        mean_degrees = 0.0

    return float(mean_degrees)


weather_clean["event_hour_utc"] = (
    pd.to_datetime(
        weather_clean["event_timestamp_utc"],
        utc=True,
        errors="coerce",
    ).dt.floor("h")
)

hourly_weather_metrics = (
    weather_clean
    .groupby(
        ["event_hour_utc", "county"],
        as_index=False,
    )
    .agg(
        weather_observation_count=(
            "event_timestamp_utc",
            "size",
        ),
        temperature_c=(
            "temperature_clean",
            "mean",
        ),
        relative_humidity_pct=(
            "relative_humidity_clean",
            "mean",
        ),
        wind_speed_mean_m_s=(
            "wind_speed_clean",
            "mean",
        ),
        wind_speed_max_m_s=(
            "wind_speed_clean",
            "max",
        ),
        wind_direction_special_count=(
            "wind_direction_is_special",
            "sum",
        ),
    )
)

hourly_wind_direction = (
    weather_clean
    .groupby(
        ["event_hour_utc", "county"],
    )["wind_direction_clean"]
    .agg(circular_mean_degrees)
    .rename("wind_direction_mean_deg")
    .reset_index()
)

hourly_weather_metrics = (
    hourly_weather_metrics
    .merge(
        hourly_wind_direction,
        on=["event_hour_utc", "county"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["event_hour_utc", "county"]
    )
    .reset_index(drop=True)
)

hourly_weather_metrics[
    "wind_direction_special_count"
] = (
    hourly_weather_metrics[
        "wind_direction_special_count"
    ].astype("int64")
)

print(
    "Full-period hourly weather rows:",
    len(hourly_weather_metrics),
)

print(
    "County-hour duplicates:",
    hourly_weather_metrics.duplicated(
        subset=["event_hour_utc", "county"]
    ).sum(),
)

print(
    "Aggregated 999 reports:",
    hourly_weather_metrics[
        "wind_direction_special_count"
    ].sum(),
)

display(
    hourly_weather_metrics.isna().sum().rename(
        "missing_count"
    ).to_frame()
)

display(hourly_weather_metrics.head())

Full-period hourly weather rows: 5805
County-hour duplicates: 0
Aggregated 999 reports: 2530


,missing_count
event_hour_utc,0
county,0
weather_observation_count,0
temperature_c,2
relative_humidity_pct,2
wind_speed_mean_m_s,21
wind_speed_max_m_s,21
wind_direction_special_count,0
wind_direction_mean_deg,1168


,event_hour_utc,county,weather_observation_count,temperature_c,relative_humidity_pct,wind_speed_mean_m_s,wind_speed_max_m_s,wind_direction_special_count,wind_direction_mean_deg
0,2021-07-13 00:00:00+00:00,Butte,1,37.200000,25.000000,4.10,4.1,0,170.000000
1,2021-07-13 00:00:00+00:00,Lassen,3,37.666667,3.333333,7.70,8.2,0,286.670437
2,2021-07-13 00:00:00+00:00,Plumas,3,37.666667,3.333333,7.70,8.2,0,286.670437
3,2021-07-13 00:00:00+00:00,Shasta,1,40.600000,8.000000,6.20,6.2,0,150.000000
4,2021-07-13 00:00:00+00:00,Tehama,2,39.400000,11.500000,5.95,6.2,0,155.000000


In [43]:
five_day_weather_sample_path = (
    project_root
    / "data"
    / "interim"
    / "noaa_ghcnh"
    / "dixie_weather_hourly_county_sample_2021-07-13_17.csv"
)

if not five_day_weather_sample_path.exists():
    raise FileNotFoundError(
        "The validated five-day NOAA interim file was not found."
    )

validated_five_day_weather = pd.read_csv(
    five_day_weather_sample_path
)

print(
    "Validated five-day shape:",
    validated_five_day_weather.shape,
)

print("\nValidated columns in saved order:")

for column_number, column_name in enumerate(
    validated_five_day_weather.columns,
    start=1,
):
    print(f"{column_number:>2}. {column_name}")

station_metadata_columns = [
    column
    for column in validated_five_day_weather.columns
    if (
        "station" in column.lower()
        or "proxy" in column.lower()
    )
]

print("\nValidated county/station metadata:")

display(
    validated_five_day_weather[
        ["county", *station_metadata_columns]
    ]
    .drop_duplicates()
    .sort_values("county")
    .reset_index(drop=True)
)

Validated five-day shape: (600, 14)

Validated columns in saved order:
 1. event_hour_utc
 2. county
 3. weather_station_id
 4. weather_station_name
 5. station_assignment
 6. weather_observation_count
 7. temperature_c
 8. relative_humidity_percent
 9. wind_speed_m_s
10. maximum_wind_speed_m_s
11. wind_direction_degrees
12. wind_direction_special_count
13. weather_available
14. weather_proxy_used

Validated county/station metadata:


,county,weather_station_id,weather_station_name,station_assignment,weather_proxy_used
0,Butte,USW00093210,OROVILLE MUNI AP,in_county,False
1,Lassen,USI0000KSVE,SUSANVILLE MUNI,in_county,False
2,Plumas,USI0000KSVE,SUSANVILLE MUNI,nearby_proxy,True
3,Shasta,USW00024257,REDDING AP,in_county,False
4,Tehama,USW00024216,RED BLUFF MUNI AP,in_county,False


## Construct and validate the full-period NOAA weather table

The cleaned weather observations are joined to the complete 6,000-row
county-hour grid. Missing observation hours remain in the table, station
metadata is preserved, and the first five days are compared with the
previously validated feasibility output.

In [45]:
# Create the complete 1,200-hour × 5-county grid.
# This avoids relying on an undefined full_period_grid variable.

period_hours = 1200

full_period_hours = pd.date_range(
    start=pd.to_datetime(
        pilot_start_utc,
        utc=True,
    ).floor("h"),
    periods=period_hours,
    freq="h",
)

county_values = (
    validated_five_day_weather[
        "county"
    ]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

assert len(county_values) == 5
assert len(full_period_hours) == 1200

full_period_grid = (
    pd.MultiIndex.from_product(
        [
            full_period_hours,
            county_values,
        ],
        names=[
            "event_hour_utc",
            "county",
        ],
    )
    .to_frame(index=False)
)

assert len(full_period_grid) == 6000

print(
    "Created full-period county-hour grid:",
    len(full_period_grid),
)


weather_station_metadata = (
    validated_five_day_weather[
        [
            "county",
            "weather_station_id",
            "weather_station_name",
            "station_assignment",
            "weather_proxy_used",
        ]
    ]
    .drop_duplicates()
    .sort_values("county")
    .reset_index(drop=True)
)

assert len(weather_station_metadata) == 5
assert weather_station_metadata["county"].nunique() == 5


# Ensure the proxy indicator is Boolean.
if not pd.api.types.is_bool_dtype(
    weather_station_metadata["weather_proxy_used"]
):
    weather_station_metadata["weather_proxy_used"] = (
        weather_station_metadata["weather_proxy_used"]
        .astype("string")
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
            }
        )
    )

assert (
    weather_station_metadata[
        "weather_proxy_used"
    ].isna().sum()
    == 0
)


weather_metrics_for_merge = (
    hourly_weather_metrics
    .rename(
        columns={
            "relative_humidity_pct":
                "relative_humidity_percent",
            "wind_speed_mean_m_s":
                "wind_speed_m_s",
            "wind_speed_max_m_s":
                "maximum_wind_speed_m_s",
            "wind_direction_mean_deg":
                "wind_direction_degrees",
        }
    )
)


full_period_weather = (
    full_period_grid[
        [
            "event_hour_utc",
            "county",
        ]
    ]
    .copy()
)

full_period_weather["event_hour_utc"] = pd.to_datetime(
    full_period_weather["event_hour_utc"],
    utc=True,
    errors="coerce",
)


full_period_weather = (
    full_period_weather
    .merge(
        weather_station_metadata,
        on="county",
        how="left",
        validate="many_to_one",
    )
    .merge(
        weather_metrics_for_merge,
        on=[
            "event_hour_utc",
            "county",
        ],
        how="left",
        validate="one_to_one",
    )
)


# Hours without NOAA observations remain in the complete grid.
for count_column in [
    "weather_observation_count",
    "wind_direction_special_count",
]:
    full_period_weather[count_column] = (
        full_period_weather[count_column]
        .fillna(0)
        .astype("int64")
    )


weather_measurement_columns = [
    "temperature_c",
    "relative_humidity_percent",
    "wind_speed_m_s",
    "maximum_wind_speed_m_s",
    "wind_direction_degrees",
]

full_period_weather["weather_available"] = (
    full_period_weather[
        weather_measurement_columns
    ]
    .notna()
    .any(axis=1)
)


full_period_weather = (
    full_period_weather[
        [
            "event_hour_utc",
            "county",
            "weather_station_id",
            "weather_station_name",
            "station_assignment",
            "weather_observation_count",
            "temperature_c",
            "relative_humidity_percent",
            "wind_speed_m_s",
            "maximum_wind_speed_m_s",
            "wind_direction_degrees",
            "wind_direction_special_count",
            "weather_available",
            "weather_proxy_used",
        ]
    ]
    .sort_values(
        [
            "event_hour_utc",
            "county",
        ]
    )
    .reset_index(drop=True)
)


assert len(full_period_weather) == 6000

assert (
    full_period_weather.duplicated(
        subset=[
            "event_hour_utc",
            "county",
        ]
    ).sum()
    == 0
)

assert (
    full_period_weather[
        "event_hour_utc"
    ].isna().sum()
    == 0
)

assert (
    full_period_weather[
        "weather_station_id"
    ].isna().sum()
    == 0
)

assert (
    full_period_weather.groupby(
        "county"
    ).size().eq(1200).all()
)


print(
    "Complete NOAA county-hour rows:",
    len(full_period_weather),
)

print(
    "County-hour duplicates:",
    full_period_weather.duplicated(
        subset=[
            "event_hour_utc",
            "county",
        ]
    ).sum(),
)

print(
    "Hours with raw NOAA observations:",
    int(
        full_period_weather[
            "weather_observation_count"
        ].gt(0).sum()
    ),
)

print(
    "Hours without raw NOAA observations:",
    int(
        full_period_weather[
            "weather_observation_count"
        ].eq(0).sum()
    ),
)

print(
    "Weather-available hours:",
    int(
        full_period_weather[
            "weather_available"
        ].sum()
    ),
)

display(full_period_weather.head())

Created full-period county-hour grid: 6000
Complete NOAA county-hour rows: 6000
County-hour duplicates: 0
Hours with raw NOAA observations: 5805
Hours without raw NOAA observations: 195
Weather-available hours: 5804


,event_hour_utc,county,weather_station_id,weather_station_name,station_assignment,weather_observation_count,temperature_c,relative_humidity_percent,wind_speed_m_s,maximum_wind_speed_m_s,wind_direction_degrees,wind_direction_special_count,weather_available,weather_proxy_used
0,2021-07-13 00:00:00+00:00,Butte,USW00093210,OROVILLE MUNI AP,in_county,1,37.200000,25.000000,4.10,4.1,170.000000,0,True,False
1,2021-07-13 00:00:00+00:00,Lassen,USI0000KSVE,SUSANVILLE MUNI,in_county,3,37.666667,3.333333,7.70,8.2,286.670437,0,True,False
2,2021-07-13 00:00:00+00:00,Plumas,USI0000KSVE,SUSANVILLE MUNI,nearby_proxy,3,37.666667,3.333333,7.70,8.2,286.670437,0,True,True
3,2021-07-13 00:00:00+00:00,Shasta,USW00024257,REDDING AP,in_county,1,40.600000,8.000000,6.20,6.2,150.000000,0,True,False
4,2021-07-13 00:00:00+00:00,Tehama,USW00024216,RED BLUFF MUNI AP,in_county,2,39.400000,11.500000,5.95,6.2,155.000000,0,True,False


In [46]:
validated_five_day_weather_check = pd.read_csv(
    five_day_weather_sample_path,
    dtype={
        "county": "string",
        "weather_station_id": "string",
        "weather_station_name": "string",
        "station_assignment": "string",
    },
)

validated_five_day_weather_check[
    "event_hour_utc"
] = pd.to_datetime(
    validated_five_day_weather_check[
        "event_hour_utc"
    ],
    utc=True,
    errors="coerce",
)

for boolean_column in [
    "weather_available",
    "weather_proxy_used",
]:
    if not pd.api.types.is_bool_dtype(
        validated_five_day_weather_check[
            boolean_column
        ]
    ):
        validated_five_day_weather_check[
            boolean_column
        ] = (
            validated_five_day_weather_check[
                boolean_column
            ]
            .astype("string")
            .str.lower()
            .map(
                {
                    "true": True,
                    "false": False,
                }
            )
        )

full_period_five_day_weather = (
    full_period_weather[
        full_period_weather[
            "event_hour_utc"
        ].between(
            pilot_start_utc,
            five_day_end_utc,
        )
    ]
    .sort_values(
        [
            "event_hour_utc",
            "county",
        ]
    )
    .reset_index(drop=True)
)

validated_five_day_weather_check = (
    validated_five_day_weather_check[
        full_period_five_day_weather.columns
    ]
    .sort_values(
        [
            "event_hour_utc",
            "county",
        ]
    )
    .reset_index(drop=True)
)

assert len(full_period_five_day_weather) == 600

assert (
    full_period_five_day_weather[
        "weather_available"
    ].sum()
    == 600
)

assert (
    full_period_five_day_weather[
        "weather_proxy_used"
    ].sum()
    == 120
)

assert (
    full_period_five_day_weather[
        "wind_speed_m_s"
    ].isna().sum()
    == 2
)

assert (
    full_period_five_day_weather[
        "wind_direction_degrees"
    ].isna().sum()
    == 109
)

assert (
    full_period_five_day_weather[
        "wind_direction_special_count"
    ].sum()
    == 199
)

pd.testing.assert_frame_equal(
    full_period_five_day_weather,
    validated_five_day_weather_check,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

print("Five-day NOAA overlap reproduced successfully.")
print("Validated overlap rows:", len(full_period_five_day_weather))
print("Weather-available hours:", 600)
print("Proxy-station hours:", 120)
print("Missing wind-speed hours:", 2)
print("Missing wind-direction hours:", 109)
print("Special wind-direction reports:", 199)

Five-day NOAA overlap reproduced successfully.
Validated overlap rows: 600
Weather-available hours: 600
Proxy-station hours: 120
Missing wind-speed hours: 2
Missing wind-direction hours: 109
Special wind-direction reports: 199


In [47]:
full_period_weather_output_path = (
    project_root
    / "data"
    / "interim"
    / "noaa_ghcnh"
    / "dixie_weather_hourly_county_full_2021-07-13_2021-08-31.csv"
)

full_period_weather_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

full_period_weather.to_csv(
    full_period_weather_output_path,
    index=False,
)

reloaded_full_period_weather = pd.read_csv(
    full_period_weather_output_path,
    dtype={
        "county": "string",
        "weather_station_id": "string",
        "weather_station_name": "string",
        "station_assignment": "string",
    },
)

reloaded_full_period_weather[
    "event_hour_utc"
] = pd.to_datetime(
    reloaded_full_period_weather[
        "event_hour_utc"
    ],
    utc=True,
    errors="coerce",
)

for boolean_column in [
    "weather_available",
    "weather_proxy_used",
]:
    if not pd.api.types.is_bool_dtype(
        reloaded_full_period_weather[
            boolean_column
        ]
    ):
        reloaded_full_period_weather[
            boolean_column
        ] = (
            reloaded_full_period_weather[
                boolean_column
            ]
            .astype("string")
            .str.lower()
            .map(
                {
                    "true": True,
                    "false": False,
                }
            )
        )

assert len(reloaded_full_period_weather) == 6000
assert len(reloaded_full_period_weather.columns) == 14

assert (
    reloaded_full_period_weather.duplicated(
        subset=[
            "event_hour_utc",
            "county",
        ]
    ).sum()
    == 0
)

assert (
    reloaded_full_period_weather.groupby(
        "county"
    ).size().eq(1200).all()
)

weather_county_summary = (
    reloaded_full_period_weather
    .groupby("county")
    .agg(
        total_hours=(
            "event_hour_utc",
            "size",
        ),
        weather_available_hours=(
            "weather_available",
            "sum",
        ),
        raw_observation_records=(
            "weather_observation_count",
            "sum",
        ),
        missing_temperature_hours=(
            "temperature_c",
            lambda values: values.isna().sum(),
        ),
        missing_humidity_hours=(
            "relative_humidity_percent",
            lambda values: values.isna().sum(),
        ),
        missing_wind_speed_hours=(
            "wind_speed_m_s",
            lambda values: values.isna().sum(),
        ),
        missing_wind_direction_hours=(
            "wind_direction_degrees",
            lambda values: values.isna().sum(),
        ),
        special_direction_reports=(
            "wind_direction_special_count",
            "sum",
        ),
        proxy_hours=(
            "weather_proxy_used",
            "sum",
        ),
    )
    .reset_index()
)

print("Full-period NOAA output saved successfully.")
print("Output path:", full_period_weather_output_path)
print("Saved rows:", len(reloaded_full_period_weather))
print(
    "Saved columns:",
    len(reloaded_full_period_weather.columns),
)
print(
    "Total raw observation records:",
    int(
        reloaded_full_period_weather[
            "weather_observation_count"
        ].sum()
    ),
)
print(
    "Total special-direction reports:",
    int(
        reloaded_full_period_weather[
            "wind_direction_special_count"
        ].sum()
    ),
)

display(weather_county_summary)

Full-period NOAA output saved successfully.
Output path: /Users/VV/Documents/GitHub/smokewatch-replay/data/interim/noaa_ghcnh/dixie_weather_hourly_county_full_2021-07-13_2021-08-31.csv
Saved rows: 6000
Saved columns: 14
Total raw observation records: 10917
Total special-direction reports: 2530


,county,total_hours,weather_available_hours,raw_observation_records,missing_temperature_hours,missing_humidity_hours,missing_wind_speed_hours,missing_wind_direction_hours,special_direction_reports,proxy_hours
0,Butte,1200,1186,1288,15,15,18,342,373,0
1,Lassen,1200,1112,3296,88,88,89,168,615,0
2,Plumas,1200,1112,3296,88,88,89,168,615,1200
3,Shasta,1200,1194,1555,6,6,14,467,622,0
4,Tehama,1200,1200,1482,0,0,6,218,305,0


## Full-Period Three-Source Integration

The validated NASA FIRMS, EPA PM2.5, and NOAA weather outputs are
reloaded from their saved full-period files and joined on the shared
UTC county-hour key.

Before integration, each source must contain exactly 6,000 rows,
five counties, 1,200 hours per county, and no duplicate county-hour
keys. The integrated output is also compared with the previously
validated five-day feasibility dataset.

In [48]:
def locate_full_period_csv(
    interim_subdirectory,
    preferred_filename,
):
    interim_directory = (
        project_root
        / "data"
        / "interim"
        / interim_subdirectory
    )

    preferred_path = (
        interim_directory
        / preferred_filename
    )

    if preferred_path.exists():
        return preferred_path

    matching_paths = sorted(
        interim_directory.glob(
            "*full*2021-07-13*2021-08-31*.csv"
        )
    )

    if len(matching_paths) == 1:
        return matching_paths[0]

    raise FileNotFoundError(
        f"Could not uniquely locate the full-period file "
        f"in {interim_directory}. Matches: {matching_paths}"
    )


full_period_fire_path = locate_full_period_csv(
    "nasa_firms",
    "dixie_fire_hourly_county_full_2021-07-13_2021-08-31.csv",
)

full_period_pm25_path = locate_full_period_csv(
    "epa_airdata",
    "dixie_pm25_hourly_county_full_2021-07-13_2021-08-31.csv",
)

full_period_weather_path = locate_full_period_csv(
    "noaa_ghcnh",
    "dixie_weather_hourly_county_full_2021-07-13_2021-08-31.csv",
)


full_period_fire_loaded = pd.read_csv(
    full_period_fire_path
)

full_period_pm25_loaded = pd.read_csv(
    full_period_pm25_path
)

full_period_weather_loaded = pd.read_csv(
    full_period_weather_path
)


source_file_inventory = pd.DataFrame(
    [
        {
            "source": "NASA FIRMS",
            "path": str(full_period_fire_path),
            "rows": len(full_period_fire_loaded),
            "columns": len(
                full_period_fire_loaded.columns
            ),
        },
        {
            "source": "EPA PM2.5",
            "path": str(full_period_pm25_path),
            "rows": len(full_period_pm25_loaded),
            "columns": len(
                full_period_pm25_loaded.columns
            ),
        },
        {
            "source": "NOAA weather",
            "path": str(full_period_weather_path),
            "rows": len(full_period_weather_loaded),
            "columns": len(
                full_period_weather_loaded.columns
            ),
        },
    ]
)

display(source_file_inventory)

,source,path,rows,columns
0,NASA FIRMS,/Users/VV/Documents/GitHub/smokewatch-replay/d...,6000,8
1,EPA PM2.5,/Users/VV/Documents/GitHub/smokewatch-replay/d...,6000,9
2,NOAA weather,/Users/VV/Documents/GitHub/smokewatch-replay/d...,6000,14


In [49]:
integration_key_columns = [
    "event_hour_utc",
    "county",
]

pilot_counties = [
    "Butte",
    "Lassen",
    "Plumas",
    "Shasta",
    "Tehama",
]

expected_full_period_hours = pd.date_range(
    start=pd.to_datetime(
        pilot_start_utc,
        utc=True,
    ).floor("h"),
    periods=1200,
    freq="h",
)

expected_integration_grid = (
    pd.MultiIndex.from_product(
        [
            expected_full_period_hours,
            sorted(pilot_counties),
        ],
        names=integration_key_columns,
    )
    .to_frame(index=False)
    .sort_values(integration_key_columns)
    .reset_index(drop=True)
)


def prepare_integration_source(
    source_frame,
    source_name,
):
    prepared_frame = source_frame.copy()

    missing_key_columns = (
        set(integration_key_columns)
        - set(prepared_frame.columns)
    )

    if missing_key_columns:
        raise KeyError(
            f"{source_name} is missing key columns: "
            f"{sorted(missing_key_columns)}"
        )

    prepared_frame["event_hour_utc"] = (
        pd.to_datetime(
            prepared_frame["event_hour_utc"],
            utc=True,
            errors="coerce",
        )
    )

    prepared_frame["county"] = (
        prepared_frame["county"]
        .astype("string")
        .str.strip()
    )

    boolean_columns = [
        "pm25_available",
        "pm25_negative_adjusted",
        "pm25_wildfire_qualified",
        "weather_available",
        "weather_proxy_used",
    ]

    for boolean_column in boolean_columns:
        if (
            boolean_column
            in prepared_frame.columns
            and not pd.api.types.is_bool_dtype(
                prepared_frame[boolean_column]
            )
        ):
            normalized_boolean = (
                prepared_frame[boolean_column]
                .astype("string")
                .str.lower()
                .map(
                    {
                        "true": True,
                        "false": False,
                    }
                )
            )

            if normalized_boolean.notna().sum() > 0:
                prepared_frame[boolean_column] = (
                    normalized_boolean
                )

    prepared_frame = (
        prepared_frame
        .sort_values(integration_key_columns)
        .reset_index(drop=True)
    )

    assert len(prepared_frame) == 6000

    assert (
        prepared_frame[
            "event_hour_utc"
        ].isna().sum()
        == 0
    )

    assert (
        prepared_frame.duplicated(
            subset=integration_key_columns
        ).sum()
        == 0
    )

    assert (
        set(prepared_frame["county"].dropna())
        == set(pilot_counties)
    )

    assert (
        prepared_frame.groupby(
            "county"
        ).size().eq(1200).all()
    )

    source_keys = prepared_frame[
        integration_key_columns
    ].copy()

    pd.testing.assert_frame_equal(
        source_keys,
        expected_integration_grid,
        check_dtype=False,
    )

    print(
        f"{source_name}: validated "
        f"{len(prepared_frame)} county-hour rows."
    )

    return prepared_frame


full_period_fire_loaded = (
    prepare_integration_source(
        full_period_fire_loaded,
        "NASA FIRMS",
    )
)

full_period_pm25_loaded = (
    prepare_integration_source(
        full_period_pm25_loaded,
        "EPA PM2.5",
    )
)

full_period_weather_loaded = (
    prepare_integration_source(
        full_period_weather_loaded,
        "NOAA weather",
    )
)

NASA FIRMS: validated 6000 county-hour rows.
EPA PM2.5: validated 6000 county-hour rows.
NOAA weather: validated 6000 county-hour rows.


In [50]:
source_frames_for_overlap_check = {
    "NASA FIRMS": full_period_fire_loaded,
    "EPA PM2.5": full_period_pm25_loaded,
    "NOAA weather": full_period_weather_loaded,
}

source_names = list(
    source_frames_for_overlap_check
)

for first_index in range(len(source_names)):
    for second_index in range(
        first_index + 1,
        len(source_names),
    ):
        first_name = source_names[first_index]
        second_name = source_names[second_index]

        first_columns = set(
            source_frames_for_overlap_check[
                first_name
            ].columns
        )

        second_columns = set(
            source_frames_for_overlap_check[
                second_name
            ].columns
        )

        unexpected_overlap = (
            first_columns
            & second_columns
            - set(integration_key_columns)
        )

        assert not unexpected_overlap, (
            f"Unexpected overlapping columns between "
            f"{first_name} and {second_name}: "
            f"{sorted(unexpected_overlap)}"
        )


full_period_integrated = (
    full_period_fire_loaded
    .merge(
        full_period_pm25_loaded,
        on=integration_key_columns,
        how="left",
        validate="one_to_one",
    )
    .merge(
        full_period_weather_loaded,
        on=integration_key_columns,
        how="left",
        validate="one_to_one",
    )
    .sort_values(integration_key_columns)
    .reset_index(drop=True)
)


assert len(full_period_integrated) == 6000

assert (
    full_period_integrated.duplicated(
        subset=integration_key_columns
    ).sum()
    == 0
)

assert (
    full_period_integrated.groupby(
        "county"
    ).size().eq(1200).all()
)

print(
    "Integrated full-period rows:",
    len(full_period_integrated),
)

print(
    "Integrated columns:",
    len(full_period_integrated.columns),
)

print(
    "Duplicate county-hour keys:",
    full_period_integrated.duplicated(
        subset=integration_key_columns
    ).sum(),
)

Integrated full-period rows: 6000
Integrated columns: 27
Duplicate county-hour keys: 0


In [51]:
preservation_checks = [
    (
        "fire_detection_count",
        full_period_fire_loaded,
    ),
    (
        "pm25_available",
        full_period_pm25_loaded,
    ),
    (
        "pm25_negative_adjusted",
        full_period_pm25_loaded,
    ),
    (
        "weather_available",
        full_period_weather_loaded,
    ),
    (
        "weather_proxy_used",
        full_period_weather_loaded,
    ),
    (
        "wind_direction_special_count",
        full_period_weather_loaded,
    ),
]

for metric_name, source_frame in preservation_checks:
    assert metric_name in source_frame.columns
    assert metric_name in full_period_integrated.columns

    source_total = source_frame[
        metric_name
    ].fillna(0).sum()

    integrated_total = full_period_integrated[
        metric_name
    ].fillna(0).sum()

    assert np.isclose(
        source_total,
        integrated_total,
    )

    print(
        f"{metric_name}: preserved total "
        f"{integrated_total}"
    )


full_period_integration_summary = (
    full_period_integrated
    .groupby("county")
    .agg(
        total_hours=(
            "event_hour_utc",
            "size",
        ),
        total_fire_detections=(
            "fire_detection_count",
            "sum",
        ),
        fire_detection_hours=(
            "fire_detection_count",
            lambda values: values.gt(0).sum(),
        ),
        pm25_available_hours=(
            "pm25_available",
            "sum",
        ),
        weather_available_hours=(
            "weather_available",
            "sum",
        ),
        weather_proxy_hours=(
            "weather_proxy_used",
            "sum",
        ),
    )
    .reset_index()
)

display(full_period_integration_summary)

fire_detection_count: preserved total 53394
pm25_available: preserved total 4791
pm25_negative_adjusted: preserved total 1
weather_available: preserved total 5804
weather_proxy_used: preserved total 1200
wind_direction_special_count: preserved total 2530


,county,total_hours,total_fire_detections,fire_detection_hours,pm25_available_hours,weather_available_hours,weather_proxy_hours
0,Butte,1200,1090,44,1200,1186,0
1,Lassen,1200,10107,80,0,1112,0
2,Plumas,1200,36538,142,1200,1112,1200
3,Shasta,1200,3766,54,1197,1194,0
4,Tehama,1200,1893,66,1194,1200,0


In [52]:
five_day_integrated_sample_path = (
    project_root
    / "data"
    / "interim"
    / "integrated"
    / "dixie_integrated_hourly_county_sample_2021-07-13_17.csv"
)

if not five_day_integrated_sample_path.exists():
    raise FileNotFoundError(
        "The validated five-day integrated file was not found."
    )

validated_five_day_integrated = pd.read_csv(
    five_day_integrated_sample_path
)

validated_five_day_integrated[
    "event_hour_utc"
] = pd.to_datetime(
    validated_five_day_integrated[
        "event_hour_utc"
    ],
    utc=True,
    errors="coerce",
)

full_period_five_day_integrated = (
    full_period_integrated[
        full_period_integrated[
            "event_hour_utc"
        ].between(
            pilot_start_utc,
            five_day_end_utc,
        )
    ]
    .sort_values(integration_key_columns)
    .reset_index(drop=True)
)

validated_five_day_integrated = (
    validated_five_day_integrated[
        full_period_five_day_integrated.columns
    ]
    .sort_values(integration_key_columns)
    .reset_index(drop=True)
)

assert len(full_period_five_day_integrated) == 600
assert len(validated_five_day_integrated) == 600


full_comparison = (
    full_period_five_day_integrated.copy()
)

validated_comparison = (
    validated_five_day_integrated.copy()
)

for column_name in full_comparison.columns:
    if (
        full_comparison[column_name].dtype
        == "object"
        or validated_comparison[column_name].dtype
        == "object"
        or pd.api.types.is_string_dtype(
            full_comparison[column_name]
        )
        or pd.api.types.is_string_dtype(
            validated_comparison[column_name]
        )
    ):
        full_comparison[column_name] = (
            full_comparison[column_name]
            .astype("string")
            .fillna("__MISSING__")
        )

        validated_comparison[column_name] = (
            validated_comparison[column_name]
            .astype("string")
            .fillna("__MISSING__")
        )


pd.testing.assert_frame_equal(
    full_comparison,
    validated_comparison,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

print(
    "Five-day integrated overlap reproduced successfully."
)

print(
    "Validated overlap rows:",
    len(full_period_five_day_integrated),
)

print(
    "Validated overlap columns:",
    len(full_period_five_day_integrated.columns),
)

Five-day integrated overlap reproduced successfully.
Validated overlap rows: 600
Validated overlap columns: 27


In [53]:
full_period_integrated_output_path = (
    project_root
    / "data"
    / "interim"
    / "integrated"
    / "dixie_integrated_hourly_county_full_2021-07-13_2021-08-31.csv"
)

full_period_integrated_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

full_period_integrated.to_csv(
    full_period_integrated_output_path,
    index=False,
)

reloaded_full_period_integrated = pd.read_csv(
    full_period_integrated_output_path
)

reloaded_full_period_integrated[
    "event_hour_utc"
] = pd.to_datetime(
    reloaded_full_period_integrated[
        "event_hour_utc"
    ],
    utc=True,
    errors="coerce",
)

assert len(reloaded_full_period_integrated) == 6000

assert (
    len(reloaded_full_period_integrated.columns)
    == len(full_period_integrated.columns)
)

assert (
    reloaded_full_period_integrated.duplicated(
        subset=integration_key_columns
    ).sum()
    == 0
)

assert (
    reloaded_full_period_integrated.groupby(
        "county"
    ).size().eq(1200).all()
)

print(
    "Full-period integrated output saved successfully."
)

print(
    "Output path:",
    full_period_integrated_output_path,
)

print(
    "Saved rows:",
    len(reloaded_full_period_integrated),
)

print(
    "Saved columns:",
    len(reloaded_full_period_integrated.columns),
)

print(
    "Total fire detections:",
    int(
        reloaded_full_period_integrated[
            "fire_detection_count"
        ].sum()
    ),
)

print(
    "PM2.5-available hours:",
    int(
        reloaded_full_period_integrated[
            "pm25_available"
        ].sum()
    ),
)

print(
    "Weather-available hours:",
    int(
        reloaded_full_period_integrated[
            "weather_available"
        ].sum()
    ),
)

display(full_period_integration_summary)

Full-period integrated output saved successfully.
Output path: /Users/VV/Documents/GitHub/smokewatch-replay/data/interim/integrated/dixie_integrated_hourly_county_full_2021-07-13_2021-08-31.csv
Saved rows: 6000
Saved columns: 27
Total fire detections: 53394
PM2.5-available hours: 4791
Weather-available hours: 5804


,county,total_hours,total_fire_detections,fire_detection_hours,pm25_available_hours,weather_available_hours,weather_proxy_hours
0,Butte,1200,1090,44,1200,1186,0
1,Lassen,1200,10107,80,0,1112,0
2,Plumas,1200,36538,142,1200,1112,1200
3,Shasta,1200,3766,54,1197,1194,0
4,Tehama,1200,1893,66,1194,1200,0
